# Phase E: DWG to manifest with the trained GNN

**CV5570 · Team Visionaries · Track B (GNN)**

**You give this notebook two things: a DWG file (a file-picker appears when you run step 4) and the id of the model repo you pushed at the end of Phase C/D.** Everything else is inside the notebook or is downloaded from Hugging Face. It converts the DWG to DXF, runs the trained GNN, and writes a manifest JSON with the **same schema as the classical Track A (Stage 2a) manifest**, with `source: "cv_model"` and a `confidence` on every record.

**Settled design (from the project plan):** the model predicts, per drawing primitive, one of 12 classes and which primitives belong together (doors, windows, stairs). It predicts **no rooms**. Rooms, door sides, adjacency, corridors and wall thickness come from the *unchanged* Track A geometry core, fed with the model's walls, doors and windows instead of CAD layers and blocks. Room names come from the `<text>` items in the SVG. Deterministic geometry is never delegated to the model.

| Step | What happens | Where the code is |
|---|---|---|
| 1 | DWG to DXF with ODA File Converter | `indlabel.convert_dwgs` |
| 2 | DXF to a **label-free** SVG, with the unit factor and the origin remembered | `cvexport.py` (uses the Phase B exporter) |
| 3 | SVG to primitives, 10 m tiles with 1 m overlap, a graph per tile | `fpgraph.py` (Phase A, from Hugging Face) |
| 4 | GNN on every tile; class logits and edge logits **averaged where tiles overlap** | `gnninfer.py` |
| 5 | Head 2 threshold, connected components, length-weighted vote: instances | `gnneval.decode_instances` (from Hugging Face) |
| 6 | Predictions to the four inputs the geometry core reads: leaves, openings, room tags, scale | `cvadapter.py` |
| 7 | Track A `run()` and `build_manifest()`, unchanged, then CV provenance and confidence | `tracka_core.py` (carried in this notebook) |
| 8 | Optional: compare with a classical manifest, if you provide one | `cvcompare.py` |

The Track A geometry core (from the Stage 2a notebook) and the Phase B exporter (`indlabel.py`) are **carried inside this notebook** as `%%writefile` cells, copied without change, so nothing else has to be uploaded.

A GPU is optional: inference is a forward pass over a few hundred tiles (estimated, not measured: a few minutes on a T4, longer on CPU). The ODA converter needs a Linux runtime, which Colab is. Nothing here trains anything.

## What has and has not been verified
Checked offline on CPU before this notebook was handed over (modules only; the notebook itself and the trained checkpoint have **not** been run):
- The adapter fed **perfect labels** reproduced the classical manifest on a synthetic flat, upright and rotated 30 degrees with a far-from-origin placement: 3 of 3 rooms at IoU 1.0, 5 of 5 openings with identical widths, adjacency F1 1.0, wall count 11 against 11, identical top-level keys.
- Tiling, merging and decoding reproduced silver labels exactly on a 12-tile synthetic grid (106 primitives seen by more than one tile, 24 of 24 instances, PQ 1.0).
- A random-weight model ran through the whole chain without error.
- The embedded cells of this notebook were rebuilt into modules and re-tested the same way.

Not yet known: the DWG to DXF conversion on your file, how the real checkpoint behaves on it, and how well the text heuristics pick room names on a real drawing.

## 0. Setup
`ezdxf`, `shapely`, `svgelements` and `pydantic` are needed by the exporter, the graph builder and the geometry core. `torch`, `numpy`, `scipy`, `pandas` and `matplotlib` are already on Colab.

ODA File Converter turns DWG into DXF. This is the same install cell as in Phase B (a Linux `.deb` plus `xvfb`, because the converter wants a display). It is skipped when you give a DXF instead of a DWG.

In [ ]:
%pip install -q ezdxf shapely svgelements pydantic huggingface_hub pyarrow pandas scipy matplotlib

In [ ]:
import os, sys, json, time, shutil, importlib, subprocess
from pathlib import Path

ODA_DEB_URL = "https://www.opendesign.com/guestfiles/get?filename=ODAFileConverter_QT6_lnxX64_11dll.deb"

if shutil.which("ODAFileConverter") is None:
    subprocess.run(["curl", "-sSL", "-o", "oda.deb", ODA_DEB_URL], check=True)
    print(subprocess.run(["file", "oda.deb"], capture_output=True, text=True).stdout.strip())
    subprocess.run("apt-get -qq update && apt-get -qq install -y xvfb libgl1 libxkbcommon-x11-0 libxcb-cursor0 ./oda.deb", shell=True, check=True)
print("ODAFileConverter:", shutil.which("ODAFileConverter"))

### Config
- `HF_DATASET_REPO`: the dataset repo that holds `fpgraph.py` and `config.json`. It is prefilled with the id used by the Phase B notebook; change it if your Phase A files live elsewhere.
- `HF_OUTPUT_REPO`: the **model** repo that step 9 of the Phase C/D notebook pushed (it holds `finetune/best.pt`, `finetune/head2_threshold.json`, `gnnmodel.py`, `gnndata.py`, `gnneval.py`). This one you must set.
- `DWG_PATH = None` opens a file picker in step 4. Or give a path (for example on a mounted Drive). A `.dxf` is accepted too and skips the conversion.
- `FLOOR_BLOCK = None` explodes the INSERT whose block has the most entities, as Stage 2a does. A DWG that holds **several floors** as separate blocks needs the block name of the floor you want; step 4 lists the candidates.
- `HEAD2_THR = None` reads the threshold Phase C/D saved beside the checkpoint (0.3 in the last run); set a number to override it.
- `MM_FACTOR = None` lets the exporter infer the unit factor from door swings and wall thickness; set a number to force it.
- `REFERENCE_MANIFEST = None`: optionally the path to a classical manifest of the same floor. If given, step 8 compares the two. Nothing breaks if it is left empty.

In [ ]:
HF_DATASET_REPO = "saiashish29/FloorPlanCAD"
HF_OUTPUT_REPO = "<YOUR_HF_MODEL_REPO>"
CKPT_SUBDIR = "finetune"

DWG_PATH = None
FLOOR_BLOCK = None
FLOOR_ONLY = True
MM_FACTOR = None
HEAD2_THR = None

NODE_BUDGET = 12000
MAX_TILES = 8
USE_AMP = True

REFERENCE_MANIFEST = None
RUN_ORACLE = False
RUN_ONE_CALL = False

WORK_DIR = Path("cv_work")
OUT_PATH = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")


def require(name, value):
    if value.startswith("<"):
        raise ValueError(f"Set {name} in the config cell before running this step")


WORK_DIR.mkdir(exist_ok=True)
print("token found:", bool(HF_TOKEN))

## 1. Fetch the model and the Phase A files
Same rule as Phases B and C: the tiles must be built by the **exact** parser and graph settings the model was trained on, so `fpgraph.py` and `config.json` come from the dataset repo. From the model repo come the fine-tuned checkpoint, the saved Head 2 threshold and the three modules needed to rebuild the network (`FloorGAT(ModelConfig(**ck["cfg"]))`).

In [ ]:
from huggingface_hub import hf_hub_download

require("HF_OUTPUT_REPO", HF_OUTPUT_REPO)
for fname in ("fpgraph.py", "config.json"):
    hf_hub_download(HF_DATASET_REPO, fname, repo_type="dataset", local_dir=".", token=HF_TOKEN)
for fname in ("gnnmodel.py", "gnndata.py", "gnneval.py", f"{CKPT_SUBDIR}/best.pt", f"{CKPT_SUBDIR}/head2_threshold.json"):
    hf_hub_download(HF_OUTPUT_REPO, fname, repo_type="model", local_dir=".", token=HF_TOKEN)
sys.path.insert(0, str(Path(".").resolve()))
print("checkpoint", Path(CKPT_SUBDIR, "best.pt").stat().st_size // 1_000_000, "MB; Phase A files and model modules downloaded")

## 2. The code this notebook carries
Everything reusable is written to a `.py` file with `%%writefile`, as in the earlier phases, so the next stage (the FastAPI service) can import it. Data flow for one floor:

`DWG` -> `DXF` -> `cvexport` -> `SVG + origin` -> `fpgraph` tiles -> `gnninfer` (model, merge, decode) -> `scene` (class, probability, instance id per primitive) -> `cvadapter` -> `tracka_core.run()` -> `manifest`.

### 2a. Track A geometry core
The cells below are the **definition-only cells of the Stage 2a notebook, copied unchanged** (the cells that load a DXF, print or write files are left out). They produce `tracka_core.py`. The adapter runs this code as it is, so a room, a corridor width or an adjacency edge is computed by the same functions in both tracks. The headings are the original Stage 2a ones.

#### Track A: 1. Configuration
All drawing-specific knowledge lives in `CONFIG`. Layer names are matched **after normalisation**: bound-xref prefixes such as `GROUND FLOOR PLAN 28-09-2021$0$A-WALL` become `A-WALL`.

- `boundary_layers` are lines that enclose space (walls, columns, curtain wall, low walls, balcony edges and railings). Stairs, furniture, cut-outs and hatches are deliberately **not** boundaries, otherwise every stair tread would become a "room".
- `room_tag_layers` holds the room tag blocks. In this drawing the `RD` block has attributes `RT` (room name) and `SZ` (designed clear size such as `3650X4170`, or `1650 WIDE` for corridors). `SZ` becomes free ground truth for validation.
- `annotation_label_regex` holds tag texts that are notes rather than rooms (`UP`, `DN`, `CUT-OUT`, `13 PASS.`, ...).
- All tolerances are in **mm**. Geometry is normalised to mm at load time, whatever `$INSUNITS` says.

For a new drawing, edit only this cell. The layer report in step 2 shows you what the drawing contains.

In [ ]:
%%writefile tracka_core.py
import re
import json
import math
import time
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import ezdxf
from ezdxf import path as ezpath
from ezdxf.math import Matrix44, Vec3
from shapely import STRtree
from shapely.geometry import Point, LineString, Polygon, MultiPolygon, box
from shapely.ops import unary_union
import shapely

CONFIG = {
    "floor_block": None,
    "boundary_layers": {"A-WALL", "A-COL", "A-CW", "A-LOW WALL", "A-RET-WALL", "A-BALCONY", "A-RAIL"},
    "wall_layers": {"A-WALL"},
    "door_layers": {"A-DOOR", "A-DOOR-INT", "N-DOOR"},
    "window_layers": {"A-WIN"},
    "room_tag_layers": {"A-RD"},
    "room_tag_name_attr": "RT",
    "room_tag_size_attr": "SZ",
    "room_text_layers": set(),
    "opening_tag_layers": {"A-TAG-DOOR"},
    "level_tag_attrs": {("LVLNAME", "LVLVALUE"), ("LVL", "%%P0'0\"")},
    "unit_label_layers": {"A-UNIT NO"},
    "annotation_label_regex": r"^(UP|DN|DOWN|RAMP( UP| DN| DOWN)?|\d+\s*PASS\.?)$|CUT-OUT|DOUBLE HT|SLAB|RCC ROOF|^@|JALI|BOXING|^(FHC|ELE|COM)-?\d+",
    "open_space_closing": 2000.0,
    "split_gap_range": (450.0, 2600.0),
    "split_angle_tol_deg": 5.0,
    "corridor_label_regex": r"CORRIDOR|PASSAGE|LOBBY",
    "circulation_label_regex": r"STAIR\w*|FIRE TOWER|FIRE LIFT|LIFT|RAMP",
    "default_insunits": 4,
    "snap_tol": 15.0,
    "flatten_tol": 5.0,
    "min_hole_area": 50_000.0,
    "void_area": 1_000_000.0,
    "void_width": 300.0,
    "wall_thickness_range": (50.0, 450.0),
    "wall_min_overlap": 100.0,
    "opening_search_pad": 60.0,
    "door_probe_offsets": (250.0, 450.0, 700.0, 1000.0),
    "adjacency_gap": 450.0,
    "adjacency_min_shared": 300.0,
    "opening_tag_radius": 1500.0,
    "dimension_tol": 25.0,
    "size_check_tol_pct": 5.0,
}

UNIT_TO_METRE = {1: 0.0254, 2: 0.3048, 4: 0.001, 5: 0.01, 6: 1.0, 14: 0.1}
UNIT_NAME = {0: "unitless", 1: "inch", 2: "foot", 4: "mm", 5: "cm", 6: "m", 14: "dm"}

#### Track A: 2. Load and explode the floor block
The DXF from Stage 1 holds one floor as a **block**: modelspace has a single big `INSERT` (here `GROUND FLOOR PLAN 28-09-2021`) plus a few annotation inserts. `pick_floor_insert` chooses the insert whose block has the most entities.

`explode` walks the block recursively with `virtual_entities()`, so every coordinate ends up in world coordinates (WCS) with all nested insert transforms applied. For each entity we keep:
- `layer`: its normalised layer, where layer `0` inherits the parent insert's layer, as AutoCAD does when displaying.
- `owner_layer`: the layer of the **top-level** insert it came from. This is how we know that an arc inside a `D1-800` block belongs to a door (owner `A-DOOR`) and is not a wall.
- the geometry, flattened to a point list (arcs become polylines with a 5 mm chord tolerance).

Inserts are also recorded with their transform matrix and attributes, because doors, windows and room tags are all inserts.

In [ ]:
%%writefile -a tracka_core.py


def norm_layer(name):
    return name.split("$0$")[-1].split("|")[-1].strip().upper()


def pick_floor_insert(doc, block_name=None):
    inserts = list(doc.modelspace().query("INSERT"))
    if block_name:
        return next(i for i in inserts if i.dxf.name == block_name)
    return max(inserts, key=lambda i: len(doc.blocks[i.dxf.name]))


def short_block(name):
    return name.split("$0$")[-1]


@dataclass
class Leaf:
    dxftype: str
    layer: str
    owner_layer: str
    owner_block: str
    points: list
    closed: bool
    radius: float = 0.0


@dataclass
class InsertRec:
    name: str
    layer: str
    owner_layer: str
    matrix: Matrix44
    insert: tuple
    scale: tuple
    rotation: float
    attribs: dict
    depth: int


@dataclass
class Exploded:
    leaves: list = field(default_factory=list)
    inserts: list = field(default_factory=list)
    texts: list = field(default_factory=list)
    dimensions: list = field(default_factory=list)


def to_points(entity, tol):
    try:
        p = ezpath.make_path(entity)
    except Exception:
        return None
    pts = [(v.x, v.y) for v in p.flattening(tol)]
    return pts if len(pts) >= 2 else None


def explode(entities, out, cfg, parent_layer="0", owner_layer="", owner_block="", depth=0):
    for e in entities:
        t = e.dxftype()
        layer = norm_layer(e.dxf.layer)
        if layer == "0":
            layer = parent_layer
        if t == "INSERT":
            attribs = {a.dxf.tag: a.dxf.text for a in e.attribs}
            out.inserts.append(InsertRec(
                short_block(e.dxf.name), layer, owner_layer, e.matrix44(),
                tuple(e.dxf.insert)[:2], (e.dxf.xscale, e.dxf.yscale), e.dxf.rotation, attribs, depth))
            try:
                children = list(e.virtual_entities())
            except Exception:
                continue
            explode(children, out, cfg, layer, owner_layer or layer, owner_block or short_block(e.dxf.name), depth + 1)
        elif t in ("TEXT", "MTEXT"):
            text = e.plain_text() if t == "MTEXT" else e.dxf.text
            out.texts.append({"text": text.strip(), "layer": layer, "owner_layer": owner_layer,
                              "insert": tuple(e.dxf.insert)[:2]})
        elif t == "DIMENSION":
            out.dimensions.append((e, layer))
        elif t in ("LINE", "LWPOLYLINE", "POLYLINE", "ARC", "CIRCLE", "ELLIPSE", "SPLINE"):
            pts = to_points(e, cfg["flatten_tol"])
            if pts is None:
                continue
            closed = t == "CIRCLE" or (t in ("LWPOLYLINE", "POLYLINE") and e.closed)
            radius = e.dxf.radius if t in ("ARC", "CIRCLE") else 0.0
            out.leaves.append(Leaf(t, layer, owner_layer, owner_block, pts, closed, radius))


def load_floor(path, cfg):
    doc = ezdxf.readfile(path)
    top = pick_floor_insert(doc, cfg["floor_block"])
    scale = resolve_scale(doc, top, cfg)
    k = scale["to_metre"] * 1000.0
    entities = list(top.virtual_entities())
    if not math.isclose(k, 1.0):
        m = Matrix44.scale(k, k, k)
        for e in entities:
            try:
                e.transform(m)
            except Exception:
                scale["issues"].append(f"could not rescale {e.dxftype()}")
    scale["normalised_to_mm_factor"] = k
    out = Exploded()
    explode(entities, out, cfg)
    return doc, top, out, scale

In [ ]:
%%writefile -a tracka_core.py


def parse_nominal_width(name):
    m = re.search(r"-\s*(\d{3,4})(?:\s*X\s*(\d{3,4}))?", name.upper())
    if not m:
        return None, None
    w = float(m.group(1))
    h = float(m.group(2)) if m.group(2) else None
    return (w if 400 <= w <= 6000 else None), h


def parse_size(sz):
    s = (sz or "").upper()
    m = re.search(r"(\d{3,5})\s*[X\*]\s*(\d{3,5})", s)
    if m:
        return {"dims": sorted([float(m.group(1)), float(m.group(2))])}
    m = re.search(r"(\d{3,5})\s*M{0,2}\s*WIDE", s)
    if m:
        return {"width": float(m.group(1))}
    return None


def resolve_scale(doc, top, cfg):
    insunits = doc.header.get("$INSUNITS", 0) or cfg["default_insunits"]
    block_units = doc.blocks[top.dxf.name].block_record.dxf.get("units", 0)
    sx, sy = top.dxf.xscale, top.dxf.yscale
    issues = []
    if not math.isclose(abs(sx), abs(sy), rel_tol=1e-6):
        issues.append(f"non-uniform floor insert scale {sx}x{sy}")
    if block_units and block_units != insunits:
        issues.append(f"block units {block_units} differ from drawing units {insunits}")
    return {
        "insunits": insunits,
        "insunits_source": "header" if doc.header.get("$INSUNITS", 0) else "config_default",
        "unit_name": UNIT_NAME.get(insunits, str(insunits)),
        "to_metre": UNIT_TO_METRE.get(insunits, 1.0),
        "floor_insert_scale": [sx, sy],
        "block_units": block_units,
        "dimlfac_header": doc.header.get("$DIMLFAC", 1.0),
        "issues": issues,
        "cross_checks": {},
    }

#### Track A: 4. Classify geometry into boundaries, walls and window glazing
Each leaf entity is routed by its `owner_layer` first, then its own layer:
- Anything inside a **door** insert is skipped as a boundary. Leaf and swing arcs would carve wedges out of rooms, so doors are sealed geometrically in step 5 instead.
- Geometry inside **window** inserts is kept. Glazing lines span the opening and so seal it naturally.
- Geometry on `boundary_layers` encloses space. The subset on `wall_layers` is also used for wall-thickness pairing in step 8.

`segments_of` breaks polylines into straight segments `(x0, y0, x1, y1)` as a NumPy array, the form used by all the vector maths below.

In [ ]:
%%writefile -a tracka_core.py


def leaf_line(leaf):
    pts = leaf.points + ([leaf.points[0]] if leaf.closed else [])
    return LineString(pts)


def classify_leaves(ex, cfg):
    opening_layers = cfg["door_layers"] | cfg["window_layers"]
    walls, boundary, windows = [], [], []
    for lf in ex.leaves:
        if lf.owner_layer in opening_layers:
            if lf.owner_layer in cfg["window_layers"]:
                windows.append(lf)
            continue
        if lf.layer in cfg["boundary_layers"] or lf.owner_layer in cfg["boundary_layers"]:
            boundary.append(lf)
            if lf.layer in cfg["wall_layers"]:
                walls.append(lf)
    return walls, boundary, windows


def segments_of(leaves):
    segs = []
    for lf in leaves:
        pts = lf.points + ([lf.points[0]] if lf.closed else [])
        for a, b in zip(pts[:-1], pts[1:]):
            if math.dist(a, b) > 1e-6:
                segs.append((a[0], a[1], b[0], b[1]))
    return np.array(segs, dtype=float).reshape(-1, 4)

#### Track A: 5. Openings: doors and windows
For every door or window insert directly in the floor block:

1. **Footprint.** Take the block definition's bounding box in block coordinates and transform its corners by the insert matrix. This gives an oriented box in WCS, plus the block's local x-axis `u`, which runs along the wall, and y-axis `v`, normal to the wall. Mirrored inserts (negative x-scale) are handled by the matrix.
2. **Jambs.** Find the boundary-segment endpoints inside the padded box. These are the wall ends either side of the opening.
3. **Clear width.** Project the jamb points onto `u` and split them into left and right groups. The clear opening is the gap between the groups. This is checked against the reference width and rejected if it falls outside 0.5–1.6×.
4. **Seal.** Connect each left jamb to the right jamb at the same wall face, giving two short bridges, one per wall face. These bridges close the gap so the room becomes an enclosed hole in step 6.
5. **Width fallbacks**, in order: jamb clear opening (confidence 1.0), then swing-arc radius (the leaf width, doubled for double-leaf blocks with two equal arcs spanning the block), then the width encoded in the block name (`SD-2400` gives 2400), then the block extent.

Trace one door: a `D1-800` block is 800×805 mm in block space. Inserted at rotation 270° it lands on a wall with 4 jamb endpoints, two per face. Projected on `u` they split 0 | 800, so the clear width is 800 mm and two bridges are added.

In [ ]:
%%writefile -a tracka_core.py


def block_info(doc, name, cache):
    if name in cache:
        return cache[name]
    blk = next((b for b in doc.blocks if short_block(b.name) == name), None)
    xs, ys, radii = [], [], []
    if blk is not None:
        for e in blk:
            if e.dxftype() == "INSERT":
                try:
                    subs = list(e.virtual_entities())
                except Exception:
                    subs = []
            else:
                subs = [e]
            for s in subs:
                if s.dxftype() == "ARC":
                    radii.append(s.dxf.radius)
                pts = to_points(s, 5.0) if s.dxftype() not in ("ATTDEF", "TEXT", "MTEXT", "HATCH", "INSERT") else None
                if pts:
                    xs += [p[0] for p in pts]
                    ys += [p[1] for p in pts]
    info = {"bbox": (min(xs), min(ys), max(xs), max(ys)) if xs else None, "arc_radii": sorted(round(r, 1) for r in radii)}
    cache[name] = info
    return info


def opening_kind(rec, cfg):
    if rec.layer in cfg["door_layers"]:
        return "door"
    if rec.layer in cfg["window_layers"]:
        return "window"
    return None


def opening_subtype(name):
    m = re.match(r"\s*([A-Z]+\d?)", name.upper())
    return m.group(1) if m else name


def transform_pts(matrix, pts):
    return [(v.x, v.y) for v in matrix.transform_vertices([Vec3(x, y, 0) for x, y in pts])]


def build_openings(doc, ex, wall_segs, cfg):
    cache = {}
    endpoints = np.vstack([wall_segs[:, :2], wall_segs[:, 2:]]) if len(wall_segs) else np.zeros((0, 2))
    ep_tree = STRtree([Point(p) for p in endpoints])
    openings = []
    for rec in ex.inserts:
        kind = opening_kind(rec, cfg)
        if kind is None or rec.owner_layer:
            continue
        info = block_info(doc, rec.name, cache)
        if info["bbox"] is None:
            continue
        x0, y0, x1, y1 = info["bbox"]
        corners = transform_pts(rec.matrix, [(x0, y0), (x1, y0), (x1, y1), (x0, y1)])
        origin, ux_pt, uy_pt = transform_pts(rec.matrix, [(0, 0), (1, 0), (0, 1)])
        u = np.subtract(ux_pt, origin)
        sx = float(np.linalg.norm(u))
        u = u / sx
        v = np.subtract(uy_pt, origin)
        v = v / np.linalg.norm(v)
        obb = Polygon(corners)
        pad = obb.buffer(cfg["opening_search_pad"], join_style=2)
        idx = ep_tree.query(pad, predicate="contains")
        jambs = np.unique(np.round(endpoints[idx], 1), axis=0) if len(idx) else np.zeros((0, 2))
        nominal_w, nominal_h = parse_nominal_width(rec.name)
        block_w = (x1 - x0) * sx
        leaf_w = max(info["arc_radii"]) * sx if info["arc_radii"] else None
        big = [r for r in info["arc_radii"] if r >= 0.8 * max(info["arc_radii"])] if info["arc_radii"] else []
        double_leaf = len(big) >= 2 and abs(2 * big[-1] * sx - block_w) <= 0.15 * block_w
        if double_leaf:
            leaf_w = 2 * big[-1] * sx
        clear_w, plug, centre = None, None, None
        if len(jambs) >= 2:
            proj = jambs @ u
            off = jambs @ v
            split = (proj.max() + proj.min()) / 2
            left, right = proj <= split, proj > split
            if left.any() and right.any():
                clear_w = float(proj[right].min() - proj[left].max())
                bridges = []
                for li in np.where(left)[0]:
                    ri = np.where(right)[0][np.argmin(np.abs(off[right] - off[li]))]
                    bridges.append(LineString([tuple(jambs[li]), tuple(jambs[ri])]))
                if clear_w > 0.4 * block_w:
                    plug = unary_union(bridges).buffer(cfg["snap_tol"], cap_style="flat")
                    centre = shapely.MultiPoint([tuple(p) for p in jambs]).centroid
        ref_w = nominal_w or leaf_w or block_w
        if clear_w is not None and not (0.5 * ref_w <= clear_w <= 1.6 * ref_w):
            clear_w = None
        if plug is None:
            edge = LineString(corners[:2])
            plug = edge.buffer(cfg["snap_tol"], cap_style=2)
            centre = edge.interpolate(0.5, normalized=True)
        width = clear_w or leaf_w or nominal_w or block_w
        method = "jamb_clear_opening" if clear_w else ("leaf_arc_radius" if leaf_w else ("block_name" if nominal_w else "block_extent"))
        openings.append({
            "kind": kind, "block": rec.name, "layer": rec.layer, "subtype": opening_subtype(rec.name),
            "position": tuple(map(float, centre.coords[0])), "insert": rec.insert, "rotation": rec.rotation,
            "mirrored": rec.scale[0] * rec.scale[1] < 0, "axis": u.tolist(), "normal": v.tolist(),
            "width_mm": width, "width_method": method, "width_clear_mm": clear_w, "width_leaf_mm": leaf_w,
            "width_block_mm": block_w, "width_nominal_mm": nominal_w, "height_nominal_mm": nominal_h,
            "double_leaf": double_leaf,
            "jamb_points": len(jambs), "_plug": plug, "_obb": obb,
        })
    return openings

#### Track A: 6. Room tags, wall solid and enclosed spaces
**Room tags.** Each `RD` insert gives `raw_label` from `RT` and `nominal` from `SZ`, parsed as `3650X4170` → sorted dims `[3650, 4170]`, or `1650 WIDE` → width. When `RT` is empty and `SZ` is not a size (for example `STAIRCASE-01`), `SZ` becomes the label.

**Wall solid.** Every boundary and glazing line is buffered by `snap_tol` (15 mm). This closes drafting gaps of up to 30 mm at corners. The door bridges are added and everything is unioned. The result is a set of polygons (`comps`), one per connected wall network, roughly one per building block.

**Spaces = holes.** Every interior ring of a component is an enclosed space. Islands inside it (a column, a shaft) are subtracted, and the shape is grown back by `snap_tol` with mitred corners so that its edges land exactly on the wall faces. That is why measured spans match DIMENSIONs to the millimetre in step 9.

In [ ]:
%%writefile -a tracka_core.py


def room_tags(ex, cfg):
    tags = []
    ann = re.compile(cfg["annotation_label_regex"], re.I)
    for rec in ex.inserts:
        if rec.layer not in cfg["room_tag_layers"] or not rec.attribs:
            continue
        rt = rec.attribs.get(cfg["room_tag_name_attr"], "").strip()
        sz = rec.attribs.get(cfg["room_tag_size_attr"], "").strip()
        size = parse_size(sz)
        label = rt or (sz if size is None else "")
        if not label:
            continue
        tags.append({"raw_label": label, "raw_size": sz, "nominal": size, "position": rec.insert,
                     "annotation": bool(ann.search(label))})
    for t in ex.texts:
        if t["layer"] in cfg["room_text_layers"] and t["text"]:
            tags.append({"raw_label": t["text"], "raw_size": "", "nominal": None, "position": t["insert"],
                         "annotation": bool(ann.search(t["text"]))})
    return tags

In [ ]:
%%writefile -a tracka_core.py


def build_solid(boundary, windows, openings, cfg):
    tol = cfg["snap_tol"]
    lines = [leaf_line(lf) for lf in boundary + windows]
    pieces = shapely.buffer(np.array(lines, dtype=object), tol, quad_segs=1, cap_style="square")
    plugs = [o["_plug"] for o in openings]
    return unary_union(list(pieces) + plugs)


def as_polygons(geom):
    if geom.is_empty:
        return []
    if geom.geom_type == "Polygon":
        return [geom]
    return [g for g in getattr(geom, "geoms", []) if g.geom_type == "Polygon"]


def extract_spaces(solid, cfg):
    comps = as_polygons(solid)
    comp_tree = STRtree(comps)
    spaces = []
    for ci, comp in enumerate(comps):
        for ring in comp.interiors:
            hole = Polygon(ring)
            if hole.area < cfg["min_hole_area"]:
                continue
            inner = [comps[j] for j in comp_tree.query(hole, predicate="contains")]
            shape = hole.difference(unary_union(inner)) if inner else hole
            shape = shape.buffer(cfg["snap_tol"], join_style=2).intersection(hole.buffer(cfg["snap_tol"], join_style=2))
            for poly in as_polygons(shape):
                if poly.area >= cfg["min_hole_area"]:
                    spaces.append({"component": ci, "geom": poly})
    return comps, spaces

#### Track A: 7. Assign labels, recover open spaces, split merged spaces
**Assign.** Point-in-polygon. Each tag goes to the smallest space containing it. Annotation tags become `notes`. Only components that receive at least one real tag are kept, which drops the opening-schedule elevations, key plan and legends that sit beside the plan.

**Open spaces.** Lobbies, corridors and staircases often open straight to the outside, so they are never holes. A morphological closing (grow then shrink the building footprint by `open_space_closing` = 2 m) fills mouths narrower than about 4 m. The new pockets that contain an unassigned tag become spaces with `enclosure="open"` and confidence 0.7.

**Tag-guided splitting.** A kitchen whose opening has no door block merges with the living room. For any space holding two or more tags we look for **wall gaps**: two wall endpoints 0.45–2.6 m apart, facing each other along the wall's own direction. Cuts are tried shortest first, and **a cut is kept only if it separates tags**. Real open plans with no wall between them (living/dining/foyer) therefore stay merged, which is correct. Split rooms get `enclosure="split"` and carry the cut lines for review.

**Measures per space:**
- `area_m2`, `perimeter_m`.
- `mrr_dims_mm`: the sides of the minimum rotated rectangle.
- `inscribed_width_mm`: the diameter of the largest inscribed circle.
- `clear_width_mm`: the bottleneck width. This is the largest `d` for which eroding by `d/2` still leaves a single connected piece, found by binary search. For a corridor it is the narrowest clear width a person must pass through, which is what corridor-width norms constrain.
- `size_check`: compares measured against `SZ`, and flags deltas over 5%.

Spaces narrower than 300 mm with no tag are wall cavities (`void`) and are dropped from the manifest.

In [ ]:
%%writefile -a tracka_core.py


def clear_width(poly, iters=14):
    lo, hi = 0.0, inscribed_width(poly) + 1.0
    for _ in range(iters):
        mid = (lo + hi) / 2
        g = poly.buffer(-mid / 2, join_style=2)
        if not g.is_empty and g.geom_type == "Polygon":
            lo = mid
        else:
            hi = mid
    return lo


def inscribed_width(poly):
    if hasattr(shapely, "maximum_inscribed_circle"):
        return 2 * shapely.maximum_inscribed_circle(poly, 1.0).length
    return 2 * shapely.polylabel(poly, 1.0).distance(poly.exterior)


def mrr_dims(poly):
    r = poly.minimum_rotated_rectangle
    c = list(r.exterior.coords)
    return sorted([math.dist(c[0], c[1]), math.dist(c[1], c[2])])

In [ ]:
%%writefile -a tracka_core.py


def assign_tags(spaces, tags, comps):
    tree = STRtree([s["geom"] for s in spaces])
    for s in spaces:
        s["tags"], s["notes"] = [], []
    unassigned = []
    for t in tags:
        p = Point(t["position"])
        hit = [i for i in tree.query(p, predicate="within")]
        if not hit:
            unassigned.append(t)
            continue
        i = min(hit, key=lambda k: spaces[k]["geom"].area)
        (spaces[i]["notes"] if t["annotation"] else spaces[i]["tags"]).append(t)
    return unassigned


def size_check(space, dims, area_m2, to_m, cfg):
    noms = [t["nominal"] for t in space["tags"] if t["nominal"]]
    if len(space["tags"]) == 1 and noms and "dims" in noms[0]:
        nd = noms[0]["dims"]
        delta = [100 * (m - n) / n for m, n in zip(dims, nd)]
        nominal_area = nd[0] * nd[1] * to_m * to_m
        return {"nominal_dims_mm": nd, "measured_dims_mm": [round(d) for d in dims],
                "delta_pct": [round(d, 2) for d in delta],
                "nominal_area_m2": round(nominal_area, 3),
                "area_delta_pct": round(100 * (area_m2 - nominal_area) / nominal_area, 2),
                "within_tol": all(abs(d) <= cfg["size_check_tol_pct"] for d in delta)}
    if noms and all("dims" in n for n in noms):
        nominal_area = sum(n["dims"][0] * n["dims"][1] for n in noms) * to_m * to_m
        return {"nominal_area_m2": round(nominal_area, 3),
                "area_delta_pct": round(100 * (area_m2 - nominal_area) / nominal_area, 2),
                "merged_tags": len(space["tags"]), "within_tol": None}
    return None


def describe_spaces(spaces, to_m, cfg):
    corr = re.compile(cfg["corridor_label_regex"], re.I)
    for i, s in enumerate(spaces):
        g = s["geom"]
        dims = mrr_dims(g)
        area_m2 = g.area * to_m * to_m
        cw, iw = clear_width(g), inscribed_width(g)
        labels = [t["raw_label"] for t in s["tags"]]
        if cw < cfg["void_width"] and not labels:
            status = "void"
        elif labels:
            status = "room"
        elif g.area < cfg["void_area"]:
            status = "void"
        else:
            status = "unlabelled"
        widths = [t["nominal"]["width"] for t in s["tags"] if t["nominal"] and "width" in t["nominal"]]
        s.update({
            "id": f"R{i:03d}", "status": status, "labels": labels,
            "raw_label": " / ".join(labels) if labels else None,
            "is_corridor": any(corr.search(l) for l in labels),
            "area_m2": round(area_m2, 3), "perimeter_m": round(g.length * to_m, 3),
            "mrr_dims_mm": [round(d) for d in dims], "inscribed_width_mm": round(iw), "clear_width_mm": round(cw),
            "nominal_width_mm": widths[0] if widths else None,
            "size_check": size_check(s, dims, area_m2, to_m, cfg),
        })
        if s["nominal_width_mm"]:
            s["width_check"] = {"nominal_mm": s["nominal_width_mm"], "clear_mm": round(cw), "inscribed_mm": round(iw),
                                "delta_pct": round(100 * (cw - s["nominal_width_mm"]) / s["nominal_width_mm"], 2)}
        s.setdefault("enclosure", "closed")
        conf = 1.0
        if status != "room":
            conf = 0.6
        elif s["enclosure"] == "open":
            conf = 0.7
        elif s["enclosure"] == "split":
            conf = 0.85
        elif s["size_check"] and s["size_check"].get("within_tol") is False:
            conf = 0.85
        s["confidence"] = conf

In [ ]:
%%writefile -a tracka_core.py


def open_spaces(comps, plan_comps, tags, cfg):
    g = cfg["open_space_closing"]
    filled = unary_union([Polygon(comps[c].exterior) for c in plan_comps])
    closed = filled.buffer(g, join_style=2).buffer(-g, join_style=2)
    tol = cfg["snap_tol"]
    pockets = as_polygons(closed.difference(filled).buffer(-1).buffer(1))
    found = []
    for p in pockets:
        p = p.buffer(tol, join_style=2).intersection(closed)
        p = max(as_polygons(p), key=lambda q: q.area) if as_polygons(p) else p
        if p.area >= cfg["min_hole_area"] and any(p.contains(Point(t["position"])) for t in tags if not t["annotation"]):
            found.append({"component": -1, "geom": p, "enclosure": "open"})
    return found


def endpoint_dirs(segs):
    a, b = segs[:, :2], segs[:, 2:]
    d = b - a
    n = np.hypot(d[:, 0], d[:, 1])[:, None]
    d = d / n
    return np.vstack([a, b]), np.vstack([-d, d])


def gap_cuts(geom, pts, dirs, cfg):
    lo, hi = cfg["split_gap_range"]
    cos_tol = math.cos(math.radians(cfg["split_angle_tol_deg"]))
    x0, y0, x1, y1 = geom.bounds
    m = (pts[:, 0] >= x0 - 50) & (pts[:, 0] <= x1 + 50) & (pts[:, 1] >= y0 - 50) & (pts[:, 1] <= y1 + 50)
    P, D = pts[m], dirs[m]
    if len(P) < 2:
        return []
    near = shapely.distance(geom.boundary, shapely.points(P)) <= 2 * cfg["snap_tol"] + 1
    P, D = P[near], D[near]
    PD, keep = np.unique(np.round(np.hstack([P, D * 1000]), 1), axis=0, return_index=True)
    P, D = P[keep], D[keep]
    cuts = []
    for i in range(len(P)):
        v = P - P[i]
        L = np.hypot(v[:, 0], v[:, 1])
        ok = (L >= lo) & (L <= hi)
        for j in np.where(ok)[0]:
            if j <= i or np.allclose(P[i], P[j]):
                continue
            w = v[j] / L[j]
            if D[i] @ w >= cos_tol and D[j] @ -w >= cos_tol:
                line = LineString([tuple(P[i]), tuple(P[j])])
                if geom.contains(line.interpolate(0.5, normalized=True)):
                    cuts.append(line)
    return sorted(cuts, key=lambda c: c.length)


def extend(line, d):
    (x0, y0), (x1, y1) = line.coords
    L = line.length
    ux, uy = (x1 - x0) / L, (y1 - y0) / L
    return LineString([(x0 - ux * d, y0 - uy * d), (x1 + ux * d, y1 + uy * d)])


def tag_count(poly, tags):
    return sum(poly.contains(Point(t["position"])) for t in tags)


def split_space(space, pts, dirs, cfg):
    from shapely.ops import split as shp_split
    tags = space["tags"]
    if len({tuple(t["position"]) for t in tags}) < 2:
        return [space]
    faces, used = [space["geom"]], []
    for cut in gap_cuts(space["geom"], pts, dirs, cfg):
        mid = cut.interpolate(0.5, normalized=True)
        for fi, f in enumerate(faces):
            if not f.contains(mid):
                continue
            parts = as_polygons(shp_split(f, extend(cut, 3 * cfg["snap_tol"])))
            if len(parts) >= 2 and sum(tag_count(p, tags) > 0 for p in parts) >= 2:
                faces = faces[:fi] + parts + faces[fi + 1:]
                used.append(cut)
            break
    if not used:
        return [space]
    tagged = [f for f in faces if tag_count(f, tags)]
    for f in faces:
        if not tag_count(f, tags):
            k = max(range(len(tagged)), key=lambda i: tagged[i].boundary.intersection(f.buffer(1)).length)
            tagged[k] = tagged[k].union(f)
    out = []
    for f in tagged:
        sub = dict(space)
        sub.update({"geom": f, "enclosure": "split" if space.get("enclosure", "closed") == "closed" else space["enclosure"],
                    "tags": [t for t in tags if f.buffer(1).contains(Point(t["position"]))],
                    "notes": [t for t in space["notes"] if f.buffer(1).contains(Point(t["position"]))],
                    "split_lines": [[[round(x, 1), round(y, 1)] for x, y in c.coords] for c in used if c.buffer(5).intersects(f.boundary)]})
        out.append(sub)
    return out

#### Track A: 8. Connectivity, adjacency graph and wall segments
**Door sides.** From each opening's centre we probe along the wall normal at 250, 450, 700 and 1000 mm on both sides, and record the space hit on each side. A probe that lands outside every building footprint returns `EXTERIOR`, which marks the opening `is_external` (exit and facade candidates). The nearest `TAG-DOOR-WINOW` tag (attribute `DW`, e.g. `D1`, `MD`) within 1.5 m gives each opening its schedule `type_code`.

**Adjacency.** Two spaces are adjacent if more than 300 mm of one's boundary lies within 450 mm (one wall thickness) of the other. Each edge carries `shared_wall_mm`, the doors and windows that connect it, and `passages`: the doorless openings found by the tag-guided split. A kitchen reached through an arch has no door, but it is still connected, and egress routing needs that connection. A door whose two probes land in the same space is flagged `self_loop` (confidence 0.5). That means its gap was not sealed, so treat the merged space with care. This edge list is what the TopologyGraph in Stage 4 loads into networkx.

**Wall segments.** Wall face segments are bucketed by angle (1°). Each face pairs with the nearest parallel face 50–450 mm away that overlaps it by at least 100 mm. The centreline is the midline over the overlap, and the thickness is the face separation. This produces the shared `WallSegment(start, end, thickness, source, confidence)` contract, which the CV track will also emit.

In [ ]:
%%writefile -a tracka_core.py


def connect_openings(openings, spaces, comps, cfg):
    spaces = [s for s in spaces if s["status"] != "void"]
    tree = STRtree([s["geom"] for s in spaces])
    comp_tree = STRtree([Polygon(c.exterior) for c in comps])

    def side(c, n, sign):
        for d in cfg["door_probe_offsets"]:
            p = Point(c[0] + sign * n[0] * d, c[1] + sign * n[1] * d)
            hit = list(tree.query(p, predicate="within"))
            if hit:
                return spaces[min(hit, key=lambda k: spaces[k]["geom"].area)]["id"]
        p = Point(c[0] + sign * n[0] * cfg["door_probe_offsets"][-1], c[1] + sign * n[1] * cfg["door_probe_offsets"][-1])
        return "EXTERIOR" if not len(comp_tree.query(p, predicate="within")) else None

    for o in openings:
        a = side(o["position"], o["normal"], +1)
        b = side(o["position"], o["normal"], -1)
        o["connects"] = [a, b]
        o["self_loop"] = a is not None and a == b
        o["is_external"] = "EXTERIOR" in (a, b)
        o["in_plan"] = len(comp_tree.query(Point(o["position"]).buffer(cfg["opening_search_pad"]), predicate="intersects")) > 0


def attach_opening_tags(openings, ex, cfg):
    tags = [r for r in ex.inserts if r.layer in cfg["opening_tag_layers"] and r.attribs]
    if not openings or not tags:
        return
    pts = np.array([o["position"] for o in openings])
    best = {}
    for r in tags:
        d = np.hypot(*(pts - np.array(r.insert)).T)
        k = int(d.argmin())
        if d[k] <= cfg["opening_tag_radius"] and (k not in best or d[k] < best[k][0]):
            best[k] = (d[k], next(iter(r.attribs.values())))
    for k, (d, code) in best.items():
        openings[k]["type_code"] = code


def build_adjacency(spaces, openings, cfg):
    rooms = [s for s in spaces if s["status"] != "void"]
    geoms = [s["geom"] for s in rooms]
    tree = STRtree(geoms)
    edges = {}
    gap = cfg["adjacency_gap"]
    for i, s in enumerate(rooms):
        grown = s["geom"].buffer(gap, join_style=2)
        for j in tree.query(grown, predicate="intersects"):
            if j <= i:
                continue
            shared = rooms[j]["geom"].boundary.intersection(grown).length
            if shared >= cfg["adjacency_min_shared"]:
                edges[(s["id"], rooms[j]["id"])] = {"a": s["id"], "b": rooms[j]["id"], "shared_wall_mm": round(shared), "doors": [], "windows": []}
    for o in openings:
        a, b = o["connects"]
        if not a or not b or a == b:
            continue
        key = tuple(sorted((a, b)))
        e = edges.setdefault(key, {"a": key[0], "b": key[1], "shared_wall_mm": None, "doors": [], "windows": []})
        e["doors" if o["kind"] == "door" else "windows"].append(o["id"])
    tree_all = STRtree(geoms)
    seen = set()
    for s in rooms:
        for line in s.get("split_lines", []):
            key = tuple(map(tuple, line))
            if key in seen:
                continue
            seen.add(key)
            ls = LineString(line)
            (x0, y0), (x1, y1) = line
            n = np.array([y0 - y1, x1 - x0]) / max(ls.length, 1)
            mid = np.array(ls.interpolate(0.5, normalized=True).coords[0])
            side = []
            for sign in (1, -1):
                p = Point(*(mid + sign * n * 80))
                hit = list(tree_all.query(p, predicate="within"))
                side.append(rooms[hit[0]]["id"] if hit else None)
            if None in side or side[0] == side[1]:
                continue
            k2 = tuple(sorted(side))
            e = edges.setdefault(k2, {"a": k2[0], "b": k2[1], "shared_wall_mm": None, "doors": [], "windows": []})
            e.setdefault("passages", []).append({"width_mm": round(ls.length), "line": line})
    for e in edges.values():
        e.setdefault("passages", [])
    for s in spaces:
        s["adjacent"] = sorted({e["b"] if e["a"] == s["id"] else e["a"] for e in edges.values() if s["id"] in (e["a"], e["b"])})
    return list(edges.values())

In [ ]:
%%writefile -a tracka_core.py


def pair_walls(segs, cfg):
    tmin, tmax = cfg["wall_thickness_range"]
    if not len(segs):
        return []
    d = segs[:, 2:] - segs[:, :2]
    L = np.hypot(d[:, 0], d[:, 1])
    keep = L > cfg["wall_min_overlap"]
    segs, d, L = segs[keep], d[keep], L[keep]
    u = d / L[:, None]
    ang = np.degrees(np.arctan2(u[:, 1], u[:, 0])) % 180
    bucket = np.round(ang).astype(int) % 180
    walls, used = [], set()
    for b in np.unique(bucket):
        idx = np.where((bucket == b) | (bucket == (b + 1) % 180) | (bucket == (b - 1) % 180))[0]
        own = idx[bucket[idx] == b]
        for i in own:
            ui, ni = u[i], np.array([-u[i][1], u[i][0]])
            p0 = segs[i, :2]
            off_a = (segs[idx, :2] - p0) @ ni
            off_b = (segs[idx, 2:] - p0) @ ni
            off = (off_a + off_b) / 2
            t0 = (segs[idx, :2] - p0) @ ui
            t1 = (segs[idx, 2:] - p0) @ ui
            lo = np.maximum(0, np.minimum(t0, t1))
            hi = np.minimum(L[i], np.maximum(t0, t1))
            ov = hi - lo
            ok = (np.abs(off) >= tmin) & (np.abs(off) <= tmax) & (ov >= cfg["wall_min_overlap"]) & (np.abs(off_a - off_b) < 5)
            if not ok.any():
                continue
            cand = np.where(ok)[0]
            k = cand[np.argmin(np.abs(off[cand]))]
            j = idx[k]
            key = (min(i, j), max(i, j))
            if key in used:
                continue
            used.add(key)
            t = abs(off[k])
            a = p0 + ui * lo[k] + ni * off[k] / 2
            c = p0 + ui * hi[k] + ni * off[k] / 2
            walls.append({"start": [round(a[0], 1), round(a[1], 1)], "end": [round(c[0], 1), round(c[1], 1)],
                          "thickness_mm": round(t, 1), "length_mm": round(float(ov[k]), 1)})
    return walls

#### Track A: 9. DIMENSION cross-validation and scale verdict
For each linear DIMENSION, take its two definition points and its measurement × `dimlfac`. If the dimension's midpoint lies inside a space and both definition points sit on that space's boundary (within 25 mm), cast a ray along the dimension direction. When the ray piece's ends coincide with the definition points, the dimension **spans the room**, and `delta_mm` = our computed span minus the dimensioned value. Dimension texts typed over the measured value (`text_override`) are reported separately, because an override is exactly how a drawing can say one thing and mean another.

`scale_cross_checks` then sets `scale_resolution.status`:
- `pass` when the median opening and room deltas are within 5%, every room-spanning dimension agrees within tolerance, and `dimlfac` = 1.
- `check` otherwise, with a unit-mismatch warning if the deltas exceed 50%.

In [ ]:
%%writefile -a tracka_core.py


def check_dimensions(ex, spaces, cfg):
    tree = STRtree([s["geom"] for s in spaces])
    out = []
    for e, layer in ex.dimensions:
        rec = {"layer": layer, "dimtype": e.dimtype}
        try:
            m = float(e.get_measurement())
        except Exception:
            continue
        if not isinstance(m, float):
            continue
        txt = (e.dxf.get("text", "") or "").strip()
        try:
            lfac = float(e.override().get("dimlfac", 1.0))
        except Exception:
            lfac = 1.0
        rec.update({"measurement_mm": round(m, 1), "dimlfac": lfac, "text_override": txt or None})
        num = re.findall(r"[\d.]+", txt.replace("<>", ""))
        if txt and txt not in ("<>",) and num:
            try:
                rec["override_delta_mm"] = round(float(num[0]) - m * lfac, 1)
            except ValueError:
                pass
        if e.dimtype in (0, 1) and m > 0:
            p2 = np.array(e.dxf.defpoint2)[:2]
            p3 = np.array(e.dxf.defpoint3)[:2]
            if e.dimtype == 0:
                a = math.radians(e.dxf.get("angle", 0.0))
                u = np.array([math.cos(a), math.sin(a)])
            else:
                u = (p3 - p2) / (np.linalg.norm(p3 - p2) or 1)
            mid = (p2 + p3) / 2
            t2, t3 = sorted([float(p2 @ u), float(p3 @ u)])
            for k in tree.query(Point(mid), predicate="within"):
                g = spaces[k]["geom"]
                o2, o3 = g.boundary.distance(Point(p2)), g.boundary.distance(Point(p3))
                if max(o2, o3) > cfg["dimension_tol"]:
                    continue
                ray = LineString([mid - u * 1e6, mid + u * 1e6]).intersection(g)
                pieces = [ray] if ray.geom_type == "LineString" else list(getattr(ray, "geoms", []))
                pieces = [q for q in pieces if q.geom_type == "LineString" and not q.is_empty]
                if not pieces:
                    continue
                piece = min(pieces, key=lambda q: q.distance(Point(mid)))
                ends = sorted(float(np.array(c) @ u) for c in piece.coords)
                spans_room = abs(ends[0] - t2) <= cfg["dimension_tol"] and abs(ends[-1] - t3) <= cfg["dimension_tol"]
                rec.update({"room": spaces[k]["id"], "endpoint_offsets_mm": [round(o2, 1), round(o3, 1)],
                            "spans_room": spans_room})
                if spans_room:
                    rec.update({"computed_span_mm": round(piece.length, 1),
                                "delta_mm": round(piece.length - m * lfac, 1)})
                break
        out.append(rec)
    return out

In [ ]:
%%writefile -a tracka_core.py


def scale_cross_checks(scale, spaces, openings, dims, cfg):
    tol = cfg["size_check_tol_pct"]
    door = [100 * (o["width_block_mm"] - o["width_nominal_mm"]) / o["width_nominal_mm"]
            for o in openings if o["width_nominal_mm"]]
    rooms = [s["size_check"]["area_delta_pct"] for s in spaces if s.get("size_check") and s["size_check"].get("within_tol") is not None]
    dm = [d["delta_mm"] for d in dims if d.get("spans_room")]
    overrides = [d for d in dims if d.get("override_delta_mm") not in (None, 0)]
    lfacs = sorted({d["dimlfac"] for d in dims})
    checks = {
        "opening_block_vs_name": {"n": len(door), "median_delta_pct": round(float(np.median(door)), 2) if door else None,
                                  "within_tol": int(sum(abs(d) <= tol for d in door))},
        "room_area_vs_tag_size": {"n": len(rooms), "median_delta_pct": round(float(np.median(rooms)), 2) if rooms else None,
                                  "within_tol": int(sum(abs(d) <= tol for d in rooms))},
        "dimension_vs_geometry": {"n": len(dm), "max_abs_delta_mm": round(max(map(abs, dm)), 1) if dm else None,
                                  "within_tol": int(sum(abs(d) <= cfg["dimension_tol"] for d in dm))},
        "dimension_text_overrides": len(overrides),
        "dimlfac_values": lfacs,
    }
    medians = [c["median_delta_pct"] for c in (checks["opening_block_vs_name"], checks["room_area_vs_tag_size"]) if c["median_delta_pct"] is not None]
    ok = all(abs(m) <= tol for m in medians) and (not dm or checks["dimension_vs_geometry"]["within_tol"] == len(dm)) and lfacs in ([], [1.0])
    if any(abs(m) > 50 for m in medians):
        scale["issues"].append("median size deltas exceed 50%: probable unit mismatch")
    scale["cross_checks"] = checks
    scale["status"] = "pass" if ok and not scale["issues"] else "check"
    return scale

#### Track A: 10. Assemble, validate and write the manifest
`build_manifest` collects everything into the frozen schema (`schema_version` 0.2):
- `rooms`: id, status, enclosure, labels, polygon (mm, WCS), area, widths, size_check, doors, windows, adjacent, confidence.
- `openings`: kind, subtype, type_code, width plus method, connects, is_external.
- `walls`, `adjacency`, `corridors` (bottleneck widths against nominal), `vertical_circulation` (stairs, lifts, fire towers, ramps).
- `dimensions`, and `unassigned_labels`, which are tags we could not place in an enclosed space and go to the human review queue.
- `scale_resolution` and `summary`.

The Pydantic models enforce the contract (`source` ∈ {deterministic, cv_model}, typed fields) before anything is written, so Stage 3 and Stage 4 can trust the shape of the data.

In [ ]:
%%writefile -a tracka_core.py


def levels_of(ex, cfg):
    lv = []
    for r in ex.inserts:
        for name_k, val_k in cfg["level_tag_attrs"]:
            if name_k in r.attribs and val_k in r.attribs:
                lv.append({"name": r.attribs[name_k], "value": r.attribs[val_k], "position": [round(c) for c in r.insert]})
    return lv


def unit_labels(ex, spaces, cfg):
    tree = STRtree([s["geom"] for s in spaces])
    out = []
    for t in ex.texts:
        if t["layer"] in cfg["unit_label_layers"] and t["text"]:
            hit = list(tree.query(Point(t["insert"]), predicate="within"))
            out.append({"label": t["text"], "position": [round(c) for c in t["insert"]],
                        "room": spaces[hit[0]]["id"] if hit else None})
    return out


def clean(poly):
    return shapely.remove_repeated_points(poly.simplify(0.5), 0.5)


def rnd_coords(poly):
    return [[round(x, 1), round(y, 1)] for x, y in clean(poly).exterior.coords]


def room_record(s, openings, edges):
    rec = {k: s[k] for k in ("id", "status", "enclosure", "raw_label", "labels", "is_corridor", "area_m2", "perimeter_m",
                             "mrr_dims_mm", "inscribed_width_mm", "clear_width_mm", "nominal_width_mm",
                             "size_check", "adjacent", "confidence")}
    rec["width_check"] = s.get("width_check")
    rec["building_component"] = s["component"]
    rec["centroid"] = [round(c, 1) for c in s["geom"].centroid.coords[0]]
    rec["bbox"] = [round(c, 1) for c in s["geom"].bounds]
    rec["polygon"] = rnd_coords(s["geom"])
    rec["holes"] = [[[round(x, 1), round(y, 1)] for x, y in r.coords] for r in clean(s["geom"]).interiors]
    rec["notes"] = [t["raw_label"] for t in s["notes"]]
    rec["raw_sizes"] = [t["raw_size"] for t in s["tags"] if t["raw_size"]]
    rec["doors"] = [o["id"] for o in openings if o["kind"] == "door" and s["id"] in o["connects"]]
    rec["open_passages"] = sorted({e["b"] if e["a"] == s["id"] else e["a"] for e in edges if e.get("passages") and s["id"] in (e["a"], e["b"])})
    rec["windows"] = [o["id"] for o in openings if o["kind"] == "window" and s["id"] in o["connects"]]
    rec["split_lines"] = s.get("split_lines", [])
    rec["source"] = "deterministic"
    return rec


def opening_record(o):
    keys = ("id", "kind", "subtype", "type_code", "block", "layer", "position", "rotation", "mirrored", "axis", "double_leaf", "self_loop",
            "width_mm", "width_method", "width_clear_mm", "width_leaf_mm", "width_block_mm", "width_nominal_mm",
            "height_nominal_mm", "connects", "is_external", "jamb_points", "confidence")
    rec = {k: o.get(k) for k in keys}
    for k in ("position", "axis"):
        rec[k] = [round(c, 4 if k == "axis" else 1) for c in rec[k]]
    for k in ("width_mm", "width_clear_mm", "width_leaf_mm", "width_block_mm"):
        if rec[k] is not None:
            rec[k] = round(rec[k], 1)
    rec["source"] = "deterministic"
    return rec

In [ ]:
%%writefile -a tracka_core.py


def corridor_summary(spaces):
    out = []
    for s in spaces:
        if s["is_corridor"]:
            g = s["geom"]
            out.append({"room": s["id"], "label": s["raw_label"], "enclosure": s["enclosure"],
                        "clear_width_mm": s["clear_width_mm"], "inscribed_width_mm": s["inscribed_width_mm"],
                        "clear_width_reliable": s["enclosure"] != "open",
                        "nominal_width_mm": s["nominal_width_mm"], "approx_length_mm": round(g.area / max(s["clear_width_mm"], 1)),
                        "area_m2": s["area_m2"]})
    return out


def circulation_summary(spaces, cfg):
    rx = re.compile(cfg["circulation_label_regex"], re.I)
    return [{"room": s["id"], "label": s["raw_label"], "kind": rx.search(s["raw_label"]).group(0).upper(),
             "area_m2": s["area_m2"], "centroid": [round(c, 1) for c in s["geom"].centroid.coords[0]]}
            for s in spaces if s["raw_label"] and rx.search(s["raw_label"])]

In [ ]:
%%writefile -a tracka_core.py


from typing import Optional, Literal
from pydantic import BaseModel, ConfigDict


class Rec(BaseModel):
    model_config = ConfigDict(extra="allow")


class RoomM(Rec):
    id: str
    status: Literal["room", "unlabelled"]
    enclosure: Literal["closed", "open", "split"]
    raw_label: Optional[str]
    area_m2: float
    polygon: list[list[float]]
    doors: list[str]
    windows: list[str]
    adjacent: list[str]
    source: Literal["deterministic", "cv_model"]
    confidence: float


class OpeningM(Rec):
    id: str
    kind: Literal["door", "window"]
    width_mm: float
    width_method: str
    connects: list[Optional[str]]
    is_external: bool
    source: Literal["deterministic", "cv_model"]
    confidence: float


class WallM(Rec):
    id: str
    start: list[float]
    end: list[float]
    thickness_mm: float
    source: Literal["deterministic", "cv_model"]
    confidence: float


class EdgeM(Rec):
    a: str
    b: str
    shared_wall_mm: Optional[float]
    doors: list[str]
    windows: list[str]


class ManifestM(Rec):
    schema_version: str
    source_file: str
    floor: dict
    units: dict
    scale_resolution: dict
    rooms: list[RoomM]
    openings: list[OpeningM]
    walls: list[WallM]
    adjacency: list[EdgeM]
    corridors: list[dict]
    vertical_circulation: list[dict]
    dimensions: list[dict]
    unassigned_labels: list[dict]
    summary: dict


def build_manifest(path, r, cfg):
    rooms = [s for s in r["spaces"] if s["status"] != "void"]
    room_ids = {s["id"] for s in rooms}
    for e in r["edges"]:
        e["doors"] = sorted(set(e["doors"]))
        e["windows"] = sorted(set(e["windows"]))
    manifest = {
        "schema_version": "0.2",
        "source_file": Path(path).name,
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "floor": {"block": r["top"].dxf.name, "insert": [round(c, 1) for c in tuple(r["top"].dxf.insert)[:2]],
                  "levels": levels_of(r["ex"], cfg), "unit_labels": unit_labels(r["ex"], rooms, cfg),
                  "building_components": len({s["component"] for s in rooms})},
        "units": {"length": "mm", "area": "m2", "coordinates": "WCS, normalised to mm"},
        "scale_resolution": r["scale"],
        "rooms": [room_record(s, r["openings"], r["edges"]) for s in rooms],
        "openings": [opening_record(o) for o in r["openings"]],
        "walls": r["walls"],
        "adjacency": [e for e in r["edges"] if e["a"] in room_ids | {"EXTERIOR"} and e["b"] in room_ids | {"EXTERIOR"}],
        "corridors": corridor_summary(rooms),
        "vertical_circulation": circulation_summary(rooms, cfg),
        "dimensions": r["dims"],
        "unassigned_labels": [{"raw_label": t["raw_label"], "raw_size": t["raw_size"], "position": [round(c, 1) for c in t["position"]]}
                              for t in r["unassigned"] if not t["annotation"]],
        "summary": {
            "rooms": sum(s["status"] == "room" for s in rooms),
            "unlabelled_spaces": sum(s["status"] == "unlabelled" for s in rooms),
            "voids_dropped": sum(s["status"] == "void" for s in r["spaces"]),
            "enclosure": dict(Counter(s["enclosure"] for s in rooms)),
            "doors": sum(o["kind"] == "door" for o in r["openings"]),
            "windows": sum(o["kind"] == "window" for o in r["openings"]),
            "external_doors": sum(o["kind"] == "door" and o["is_external"] for o in r["openings"]),
            "walls": len(r["walls"]),
            "wall_thickness_mm": dict(Counter(round(w["thickness_mm"]) for w in r["walls"]).most_common(6)),
            "adjacency_edges": len(r["edges"]),
            "unassigned_labels": sum(not t["annotation"] for t in r["unassigned"]),
            "elapsed_s": round(r["elapsed"], 1),
        },
    }
    ManifestM.model_validate(manifest)
    return manifest

#### Track A: 12. One-call version (for batch runs and the FastAPI service)
`run()` executes steps 2–9 in one call. Use `build_manifest(path, run(path), CONFIG)` from other code.

In [ ]:
%%writefile -a tracka_core.py


def run(path, cfg=CONFIG, out_path=None):
    t0 = time.time()
    doc, top, ex, scale = load_floor(path, cfg)
    to_m = 0.001
    walls_l, boundary, windows = classify_leaves(ex, cfg)
    wall_segs = segments_of(walls_l)
    bnd_segs = segments_of(boundary)
    openings = build_openings(doc, ex, bnd_segs, cfg)
    solid = build_solid(boundary, windows, openings, cfg)
    comps, spaces = extract_spaces(solid, cfg)
    tags = room_tags(ex, cfg)
    unassigned = assign_tags(spaces, tags, comps)
    tagged_comps = {s["component"] for s in spaces if s["tags"]}
    plan_zone = unary_union([Polygon(comps[c].exterior) for c in tagged_comps]).buffer(5000)
    plan_comps = [i for i, c in enumerate(comps) if c.intersects(plan_zone)]
    pockets = open_spaces(comps, plan_comps, unassigned, cfg)
    spaces = [s for s in spaces if s["component"] in tagged_comps] + pockets
    unassigned = assign_tags(spaces, tags, comps)
    pts, dirs = endpoint_dirs(bnd_segs)
    spaces = [part for s in spaces for part in split_space(s, pts, dirs, cfg)]
    describe_spaces(spaces, to_m, cfg)
    connect_openings(openings, spaces, comps, cfg)
    openings = [o for o in openings if o["in_plan"]]
    for i, o in enumerate(openings):
        o["id"] = f"{o['kind'][0].upper()}{i:03d}"
        o["confidence"] = 0.5 if o["self_loop"] else (1.0 if o["width_method"] == "jamb_clear_opening" else 0.85)
    attach_opening_tags(openings, ex, cfg)
    edges = build_adjacency(spaces, openings, cfg)
    walls = [w for w in pair_walls(wall_segs, cfg) if plan_zone.contains(Point(w["start"]))]
    for i, w in enumerate(walls):
        w.update({"id": f"W{i:04d}", "source": "deterministic", "confidence": 1.0})
    dims = check_dimensions(ex, spaces, cfg)
    scale_cross_checks(scale, spaces, openings, dims, cfg)
    return dict(doc=doc, top=top, ex=ex, scale=scale, spaces=spaces, openings=openings, edges=edges,
                walls=walls, dims=dims, tags=tags, unassigned=unassigned, comps=comps, tagged_comps=tagged_comps,
                plan_comps=plan_comps, plan_zone=plan_zone,
                elapsed=time.time() - t0)

### 2b. Phase B exporter and labeller
The next cells are `indlabel.py` from the Phase B notebook, **copied unchanged**. From it the CV track uses the DXF reader, the entity explosion, the unit inference (door swings and wall thickness) and the SVG writer, so the model sees exactly the geometry it was trained on. The labeller in the same file is used only for the optional agreement check and the oracle (steps 6 and 7), never as input to the model.

In [ ]:
%%writefile indlabel.py
import re
import math
import json
import time
import shutil
import subprocess
from dataclasses import dataclass, field, asdict
from collections import Counter, defaultdict
from pathlib import Path as FsPath
from xml.sax.saxutils import escape

import numpy as np
import pandas as pd
import ezdxf
from ezdxf import recover
from ezdxf import path as ezpath
from ezdxf.math import Vec3
import shapely
from shapely import STRtree
from shapely.geometry import LineString
from shapely.ops import unary_union

UNIT_TO_MM = {1: 25.4, 2: 304.8, 4: 1.0, 5: 10.0, 6: 1000.0, 14: 100.0}
UNIT_CANDIDATES = [1.0, 10.0, 25.4, 304.8, 1000.0]


@dataclass
class Rules:
    exclude: str = r"TEXT|TXT|DIM|TAG|ANNO|HATCH|\bHAT\b|PATT|SCHED|TITLE|GRID|AXIS|CENT(ER|RE)|LEADER|NOTE|SYMB|DEFPOINTS|VPORT|BORDER|FRAME|FURN|\bFUR\b|FUR-|FIXT|SANIT|LAND|TREE|PLANT|ELEC|PLUMB|DRAIN|HVAC|KEY\s*PLAN|LOCATION\s*PLAN|VICINITY"
    door: str = r"DOOR|(^|[-_ ])DR($|[-_ \d])|GATE|SHUTTER|^[A-Z]?D\d*\s*-\s*\d|^(MD|FD|SD|GD)\b|TOIDOR"
    window: str = r"WIN(DOW)?|VENT|LOUV|JALI|^[A-Z]?W\d*\s*-\s*\d|^(KW|BW|GW|SW|CW)\s*-?\s*\d|^V\s*-\s*\d"
    curtain: str = r"CURTAIN|GLAZ|(^|[-_ ])CW($|[-_ ])|GLASS"
    railing: str = r"RAIL|BALUSTR|GRILL"
    stair: str = r"STAIR|\bSTEPS?\b|STRCASE"
    lift: str = r"LIFT|ELEV|EVTR"
    column: str = r"(^|[^A-Z])COL(UMN|UMNS|S)?($|[^A-Z])|COLU"
    wall: str = r"WALL|BRICK|MASONRY|PARTITION|BLOCKWORK|MURO"
    bay: str = r"BAY|^BW"
    rolling: str = r"SHUTTER|ROLL"
    folding: str = r"FOLD"
    revolving: str = r"REVOLV"
    column_as: str = "wall"

    def compiled(self):
        return {k: re.compile(v, re.I) for k, v in asdict(self).items() if k != "column_as"}


CATEGORY_ORDER = ["door", "window", "curtain", "railing", "stair", "lift", "column", "wall"]
CARRIER_CATEGORIES = {"door", "window", "stair", "lift"}
STRONG_CARRIERS = {"door", "window"}
INSTANCE_SIZE_MM = {"door": (450.0, 6000.0), "window": (300.0, 10000.0), "stair": (1000.0, 40000.0), "lift": (900.0, 8000.0)}


def norm_name(name):
    return (name or "").split("$0$")[-1].split("|")[-1].strip().upper()


def categorize(name, rx):
    n = norm_name(name)
    if not n or n == "0" or rx["exclude"].search(n):
        return None
    for cat in CATEGORY_ORDER:
        if rx[cat].search(n):
            return cat
    return None


@dataclass
class Prim:
    t: str
    p: list
    layer: str
    cat: object = None
    carrier: int = -1
    carrier_name: str = ""
    r: float = 0.0
    c: tuple = None
    large: bool = False
    span: float = 0.0
    sid: int = 0
    iid: int = -1


@dataclass
class TextItem:
    text: str
    x: float
    y: float
    h: float

In [ ]:
%%writefile -a indlabel.py


def _xy(v):
    return (float(v[0]), float(v[1]))


def _arc_prims(e, layer):
    ocs = e.ocs()
    c, r = e.dxf.center, float(e.dxf.radius)
    a0, a1 = float(e.dxf.start_angle), float(e.dxf.end_angle)
    span = (a1 - a0) % 360.0
    if r <= 0:
        return []
    if span < 1e-6 or span >= 359.999:
        return [Prim("circle", [_xy(ocs.to_wcs(c))], layer, r=r, c=_xy(ocs.to_wcs(c)))]
    pts = []
    for a in (a0, a0 + span / 2, a0 + span):
        q = Vec3(c.x + r * math.cos(math.radians(a)), c.y + r * math.sin(math.radians(a)), c.z)
        pts.append(_xy(ocs.to_wcs(q)))
    return [Prim("arc", pts, layer, r=r, c=_xy(ocs.to_wcs(c)), large=span > 180, span=span)]


def _circle_prims(e, layer):
    r = float(e.dxf.radius)
    if r <= 0:
        return []
    c = _xy(e.ocs().to_wcs(e.dxf.center))
    return [Prim("circle", [c], layer, r=r, c=c)]


def _bulge_segment(p1, p2, b, ocs, elev, layer):
    x1, y1 = p1
    x2, y2 = p2
    dx, dy = x2 - x1, y2 - y1
    L = math.hypot(dx, dy)
    if L < 1e-9:
        return []
    w = lambda x, y: _xy(ocs.to_wcs(Vec3(x, y, elev)))
    if abs(b) < 1e-9:
        return [Prim("line", [w(x1, y1), w(x2, y2)], layer)]
    nx, ny = dy / L, -dx / L
    s = b * L / 2
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    r = L * (1 + b * b) / (4 * abs(b))
    cx, cy = mx + nx * (s - math.copysign(r, s)), my + ny * (s - math.copysign(r, s))
    span = math.degrees(4 * math.atan(abs(b)))
    return [Prim("arc", [w(x1, y1), w(mx + nx * s, my + ny * s), w(x2, y2)], layer, r=r, c=w(cx, cy),
                 large=abs(b) > 1, span=span)]


def _lwpolyline_prims(e, layer):
    ocs, elev = e.ocs(), float(e.dxf.get("elevation", 0.0))
    pts = [(float(x), float(y), float(b)) for x, y, b in e.get_points("xyb")]
    if len(pts) < 2:
        return []
    if e.closed:
        pts.append(pts[0])
    out = []
    for (x1, y1, b), (x2, y2, _) in zip(pts[:-1], pts[1:]):
        out += _bulge_segment((x1, y1), (x2, y2), b, ocs, elev, layer)
    return out


def _polyline_prims(e, layer):
    verts = list(e.vertices)
    if len(verts) < 2:
        return []
    if e.is_2d_polyline:
        ocs, elev = e.ocs(), float(e.dxf.elevation.z) if e.dxf.hasattr("elevation") else 0.0
        pts = [(float(v.dxf.location.x), float(v.dxf.location.y), float(v.dxf.get("bulge", 0.0))) for v in verts]
        if e.is_closed:
            pts.append(pts[0])
        out = []
        for (x1, y1, b), (x2, y2, _) in zip(pts[:-1], pts[1:]):
            out += _bulge_segment((x1, y1), (x2, y2), b, ocs, elev, layer)
        return out
    if e.is_3d_polyline:
        pts = [_xy(v.dxf.location) for v in verts]
        if e.is_closed:
            pts.append(pts[0])
        return [Prim("line", [a, b], layer) for a, b in zip(pts[:-1], pts[1:]) if math.dist(a, b) > 1e-9]
    return []


def _curve_prims(e, layer):
    try:
        p = ezpath.make_path(e)
    except Exception:
        return []
    out, cur = [], _xy(p.start)
    for cmd in p.commands():
        end = _xy(cmd.end)
        kind = cmd.type.name
        if kind == "LINE_TO":
            if math.dist(cur, end) > 1e-9:
                out.append(Prim("line", [cur, end], layer))
        elif kind == "CURVE4_TO":
            out.append(Prim("cubic", [cur, _xy(cmd.ctrl1), _xy(cmd.ctrl2), end], layer))
        elif kind == "CURVE3_TO":
            q = _xy(cmd.ctrl)
            c1 = (cur[0] + 2 / 3 * (q[0] - cur[0]), cur[1] + 2 / 3 * (q[1] - cur[1]))
            c2 = (end[0] + 2 / 3 * (q[0] - end[0]), end[1] + 2 / 3 * (q[1] - end[1]))
            out.append(Prim("cubic", [cur, c1, c2, end], layer))
        cur = end
    return out


GEOMETRY = {"LINE": lambda e, l: [Prim("line", [_xy(e.dxf.start), _xy(e.dxf.end)], l)] if e.dxf.start != e.dxf.end else [],
            "ARC": _arc_prims, "CIRCLE": _circle_prims, "LWPOLYLINE": _lwpolyline_prims, "POLYLINE": _polyline_prims,
            "ELLIPSE": _curve_prims, "SPLINE": _curve_prims}
COMPOUND = {"DIMENSION", "ARC_DIMENSION", "LARGE_RADIAL_DIMENSION", "LEADER", "MLEADER", "MULTILEADER", "ACAD_TABLE"}


@dataclass
class Exploded:
    prims: list = field(default_factory=list)
    texts: list = field(default_factory=list)
    carriers: dict = field(default_factory=dict)
    errors: Counter = field(default_factory=Counter)


def _text(e):
    t = e.dxftype()
    try:
        if t == "MTEXT":
            return TextItem(e.plain_text().strip(), *_xy(e.dxf.insert), float(e.dxf.get("char_height", 250.0)))
        return TextItem((e.dxf.text or "").strip(), *_xy(e.dxf.insert), float(e.dxf.get("height", 250.0)))
    except Exception:
        return None


def explode(entities, out, rx, parent_layer="0", carrier=-1, carrier_cat=None, depth=0):
    for e in entities:
        t = e.dxftype()
        layer = norm_name(e.dxf.get("layer", "0"))
        if layer == "0":
            layer = parent_layer
        if t == "INSERT":
            for a in getattr(e, "attribs", []):
                ti = _text(a)
                if ti and ti.text:
                    out.texts.append(ti)
            c_id, c_cat = carrier, carrier_cat
            if carrier < 0 or carrier_cat not in STRONG_CARRIERS:
                lc = categorize(layer, rx)
                bc = None if (rx["exclude"].search(layer) and layer != "0") else categorize(e.dxf.name, rx)
                cat = lc if lc in CARRIER_CATEGORIES else (bc if bc in CARRIER_CATEGORIES else None)
                if carrier >= 0 and cat not in STRONG_CARRIERS:
                    cat = None
                if cat:
                    c_id, c_cat = len(out.carriers), cat
                    out.carriers[c_id] = {"cat": cat, "names": f"{norm_name(e.dxf.name)} {layer}"}
            if depth < 12:
                try:
                    explode(e.virtual_entities(), out, rx, layer, c_id, c_cat, depth + 1)
                except Exception as ex:
                    out.errors[f"INSERT {type(ex).__name__}"] += 1
        elif t in ("TEXT", "MTEXT", "ATTRIB"):
            ti = _text(e)
            if ti and ti.text:
                out.texts.append(ti)
        elif t in COMPOUND:
            if depth < 12:
                try:
                    explode(e.virtual_entities(), out, rx, layer, carrier, carrier_cat, depth + 1)
                except Exception as ex:
                    out.errors[f"{t} {type(ex).__name__}"] += 1
        elif t in GEOMETRY:
            try:
                prims = GEOMETRY[t](e, layer)
            except Exception as ex:
                out.errors[f"{t} {type(ex).__name__}"] += 1
                continue
            own = categorize(layer, rx)
            if carrier >= 0 and (carrier_cat in STRONG_CARRIERS or own is None or own == carrier_cat):
                cat, car = carrier_cat, carrier
            else:
                cat, car = own, -1
            for p in prims:
                p.cat, p.carrier = cat, car
            out.prims.extend(prims)


def read_dxf(path):
    try:
        return ezdxf.readfile(str(path)), "ok"
    except Exception:
        doc, auditor = recover.readfile(str(path))
        return doc, f"recovered ({len(auditor.errors)} errors)"


def explode_file(path, rules):
    doc, status = read_dxf(path)
    out = Exploded()
    explode(doc.modelspace(), out, rules.compiled())
    return doc, out, status

In [ ]:
%%writefile -a indlabel.py


def wall_thickness_mode(prims, max_segments=2500):
    segs = np.array([p.p[0] + p.p[1] for p in prims if p.t == "line" and p.cat == "wall"], dtype=float).reshape(-1, 4)
    if len(segs) < 4:
        return None
    d = segs[:, 2:] - segs[:, :2]
    L = np.hypot(d[:, 0], d[:, 1])
    keep = np.argsort(-L)[:max_segments]
    segs, d, L = segs[keep], d[keep], L[keep]
    ok = L > 0
    segs, d, L = segs[ok], d[ok], L[ok]
    u = d / L[:, None]
    nearest = []
    for s in range(0, len(segs), 500):
        ui, ai, Li = u[s:s + 500], segs[s:s + 500, :2], L[s:s + 500]
        par = np.abs(ui[:, None, 0] * u[None, :, 1] - ui[:, None, 1] * u[None, :, 0]) < 0.035
        rel = segs[None, :, :2] - ai[:, None, :]
        rel2 = segs[None, :, 2:] - ai[:, None, :]
        off = ui[:, None, 0] * rel[..., 1] - ui[:, None, 1] * rel[..., 0]
        t0 = (ui[:, None, :] * rel).sum(-1)
        t1 = (ui[:, None, :] * rel2).sum(-1)
        ov = np.minimum(Li[:, None], np.maximum(t0, t1)) - np.maximum(0, np.minimum(t0, t1))
        cand = par & (ov > 0.3 * np.minimum(Li[:, None], L[None, :])) & (np.abs(off) > 1e-6)
        dist = np.where(cand, np.abs(off), np.inf)
        best = dist.min(1)
        nearest.extend(best[np.isfinite(best)].tolist())
    if len(nearest) < 4:
        return None
    lg = np.log(np.array(nearest))
    hist, edges = np.histogram(lg, bins=np.arange(lg.min(), lg.max() + 0.1, 0.05))
    k = int(hist.argmax())
    sel = (lg >= edges[k]) & (lg < edges[k + 1])
    return float(np.exp(np.median(lg[sel])))


def infer_units(doc, prims):
    header = int(doc.header.get("$INSUNITS", 0) or 0)
    swings = [p for p in prims if p.t == "arc" and p.cat == "door" and 20 <= p.span <= 180]
    per_carrier = defaultdict(float)
    for p in swings:
        if p.carrier >= 0:
            per_carrier[p.carrier] = max(per_carrier[p.carrier], p.r)
    radii = list(per_carrier.values()) + [p.r for p in swings if p.carrier < 0]
    if radii:
        top = float(np.quantile(radii, 0.9))
        radii = [r for r in radii if r >= 0.2 * top]
    r_med = float(np.median(radii)) if radii else None
    t_mode = wall_thickness_mode(prims)
    cands = list(dict.fromkeys(([UNIT_TO_MM[header]] if header in UNIT_TO_MM else []) + UNIT_CANDIDATES))

    def score(f):
        s = 0
        if r_med:
            s += 600 <= f * r_med <= 1300
        if t_mode:
            s += 75 <= f * t_mode <= 450
        return s

    best = max(cands, key=lambda f: (score(f), f == UNIT_TO_MM.get(header), f == 1.0))
    sc = score(best)
    evidence = (r_med is not None) + (t_mode is not None)
    status = "confirmed" if evidence and sc == evidence else ("partial" if sc else "unverified")
    return {"insunits": header, "header_mm": UNIT_TO_MM.get(header), "mm_factor": best, "status": status,
            "door_leaf_mm": round(best * r_med, 1) if r_med else None, "n_door_arcs": len(radii),
            "wall_thickness_mm": round(best * t_mode, 1) if t_mode else None,
            "header_overridden": header in UNIT_TO_MM and UNIT_TO_MM[header] != best}


def scale_to_mm(ex, f):
    for p in ex.prims:
        p.p = [(x * f, y * f) for x, y in p.p]
        p.r *= f
        if p.c is not None:
            p.c = (p.c[0] * f, p.c[1] * f)
    for t in ex.texts:
        t.x, t.y, t.h = t.x * f, t.y * f, t.h * f


def _geom(p):
    if p.t == "circle":
        return shapely.Point(p.c).buffer(p.r, quad_segs=4).exterior
    return LineString(p.p)


CLUSTER_TOL = {"door": 50.0, "window": 50.0, "stair": 300.0, "lift": 300.0}


def assign_instances(ex):
    for p in ex.prims:
        if p.carrier >= 0:
            p.iid = p.carrier
            p.carrier_name = ex.carriers[p.carrier]["names"]
    next_id = len(ex.carriers)
    loose_names = {}
    for cat, tol in CLUSTER_TOL.items():
        idx = [i for i, p in enumerate(ex.prims) if p.cat == cat and p.carrier < 0]
        if not idx:
            continue
        geoms = [_geom(ex.prims[i]) for i in idx]
        merged = unary_union([g.buffer(tol / 2) for g in geoms])
        blobs = list(merged.geoms) if hasattr(merged, "geoms") else [merged]
        tree = STRtree(blobs)
        base = next_id
        for i, g in zip(idx, geoms):
            hit = tree.query(g, predicate="intersects")
            k = int(hit[0]) if len(hit) else 0
            ex.prims[i].iid = base + k
            loose_names.setdefault(base + k, set()).add(ex.prims[i].layer)
        next_id = base + len(blobs)
        for k in range(base, next_id):
            ex.carriers[k] = {"cat": cat, "names": " ".join(sorted(loose_names.get(k, []))), "loose": True}
    for p in ex.prims:
        if p.iid >= 0 and not p.carrier_name:
            p.carrier_name = ex.carriers[p.iid]["names"]


def door_subtype(members, names, rx):
    pts = np.array([q for p in members for q in (p.p if p.t != "circle" else [p.c])], dtype=float)
    W = float(np.ptp(pts, axis=0).max()) if len(pts) else 0.0
    leaves = [p for p in members if p.t == "arc" and 20 <= p.span <= 180 and max(250.0, 0.2 * W) <= p.r <= min(1600.0, 1.15 * W)]
    centers = {(round(p.c[0] / 30), round(p.c[1] / 30)) for p in leaves}
    if len(centers) >= 2:
        return "double door", W, [p.r for p in leaves]
    if len(centers) == 1:
        return "single door", W, [p.r for p in leaves]
    if rx["rolling"].search(names):
        return "rolling door", W, []
    if rx["folding"].search(names):
        return "folding door", W, []
    if rx["revolving"].search(names):
        return "revolving door", W, []
    return "sliding door", W, []


STUFF_NAME = {"wall": "wall", "curtain": "curtain wall", "railing": "railing"}


def label_prims(ex, rules, raw_names):
    rx = rules.compiled()
    sid = {n: i + 1 for i, n in enumerate(raw_names)}
    stuff = dict(STUFF_NAME)
    if rules.column_as in sid:
        stuff["column"] = rules.column_as
    members = defaultdict(list)
    for p in ex.prims:
        if p.iid >= 0:
            members[p.iid].append(p)
    inst_class, door_info, implausible = {}, [], Counter()
    for iid, mem in members.items():
        cat = ex.carriers[iid]["cat"]
        names = ex.carriers[iid]["names"]
        pts = np.array([q for p in mem for q in (p.p if p.t != "circle" else [p.c])], dtype=float)
        size = float(np.ptp(pts, axis=0).max()) if len(pts) else 0.0
        lo, hi = INSTANCE_SIZE_MM.get(cat, (0.0, float("inf")))
        if not lo <= size <= hi:
            inst_class[iid] = None
            implausible[cat] += 1
            continue
        if cat == "door":
            name, W, leaf = door_subtype(mem, names, rx)
            door_info.append({"iid": iid, "subtype": name, "width_mm": round(W, 1), "leaf_radii_mm": [round(r, 1) for r in leaf],
                              "names": names, "loose": bool(ex.carriers[iid].get("loose"))})
        elif cat == "window":
            name = "bay window" if rx["bay"].search(names) else "window"
        elif cat == "stair":
            name = "stairs"
        elif cat == "lift":
            name = "elevator"
        else:
            name = None
        inst_class[iid] = name
    dense = {iid: k for k, iid in enumerate(sorted(i for i, n in inst_class.items() if n))}
    for p in ex.prims:
        if p.iid >= 0:
            name = inst_class.get(p.iid)
            p.sid, p.iid = (sid[name], dense[p.iid]) if name in sid else (0, -1)
        elif p.cat in stuff:
            p.sid, p.iid = sid[stuff[p.cat]], -1
        else:
            p.sid, p.iid = 0, -1
    ex.implausible = dict(implausible)
    return door_info

In [ ]:
%%writefile -a indlabel.py


def _bounds(ex):
    xs, ys = [], []
    for p in ex.prims:
        if p.t == "circle":
            xs += [p.c[0] - p.r, p.c[0] + p.r]
            ys += [p.c[1] - p.r, p.c[1] + p.r]
        else:
            xs += [q[0] for q in p.p]
            ys += [q[1] for q in p.p]
    if not xs:
        return 0.0, 0.0, 1.0, 1.0
    return min(xs), min(ys), max(xs), max(ys)


def write_svg(ex, path, margin=500.0):
    x0, y0, x1, y1 = _bounds(ex)
    x0, y0, x1, y1 = x0 - margin, y0 - margin, x1 + margin, y1 + margin
    fx = lambda x: x - x0
    fy = lambda y: y1 - y
    P = lambda q: f"{fx(q[0]):.2f} {fy(q[1]):.2f}"
    out = ['<?xml version="1.0" encoding="utf-8"?>',
           f'<svg xmlns="http://www.w3.org/2000/svg" version="1.1" viewBox="0 0 {x1 - x0:.2f} {y1 - y0:.2f}" data-units="mm">']
    for p in ex.prims:
        attrs = f' semantic-id="{p.sid}" instance-id="{p.iid}"' if p.sid else ""
        style = ' fill="none" stroke="rgb(0,0,0)" stroke-width="1"'
        if p.t == "line":
            out.append(f'<path d="M {P(p.p[0])} L {P(p.p[1])}"{style}{attrs}/>')
        elif p.t == "arc":
            s, m, e = [(fx(q[0]), fy(q[1])) for q in p.p]
            cross = (m[0] - s[0]) * (e[1] - m[1]) - (m[1] - s[1]) * (e[0] - m[0])
            sweep = 1 if cross > 0 else 0
            out.append(f'<path d="M {s[0]:.2f} {s[1]:.2f} A {p.r:.2f} {p.r:.2f} 0 {int(p.large)} {sweep} {e[0]:.2f} {e[1]:.2f}"{style}{attrs}/>')
        elif p.t == "circle":
            out.append(f'<circle cx="{fx(p.c[0]):.2f}" cy="{fy(p.c[1]):.2f}" r="{p.r:.2f}"{style}{attrs}/>')
        elif p.t == "cubic":
            a, b, c, d = p.p
            out.append(f'<path d="M {P(a)} C {P(b)} {P(c)} {P(d)}"{style}{attrs}/>')
    for t in ex.texts:
        out.append(f'<text x="{fx(t.x):.2f}" y="{fy(t.y):.2f}" font-size="{max(t.h, 1.0):.1f}">{escape(t.text)}</text>')
    out.append("</svg>")
    FsPath(path).parent.mkdir(parents=True, exist_ok=True)
    FsPath(path).write_text("\n".join(out), encoding="utf-8")
    return {"origin_mm": [round(x0, 1), round(y1, 1)], "size_mm": [round(x1 - x0, 1), round(y1 - y0, 1)]}


PREVIEW_COLORS = {"wall": "#1f4e9c", "curtain wall": "#17becf", "railing": "#8c564b", "single door": "#d62728",
                  "double door": "#ff7f0e", "sliding door": "#e377c2", "folding door": "#bcbd22", "rolling door": "#bcbd22",
                  "revolving door": "#bcbd22", "window": "#2ca02c", "bay window": "#98df8a", "stairs": "#9467bd",
                  "elevator": "#7f7f7f"}


def _poly(p):
    if p.t == "circle":
        a = np.linspace(0, 2 * np.pi, 13)
        return np.stack([p.c[0] + p.r * np.cos(a), p.c[1] + p.r * np.sin(a)], 1)
    return np.array(p.p)


def preview(ex, raw_names, path, title=""):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    fig, ax = plt.subplots(figsize=(14, 14))
    groups = defaultdict(list)
    for p in ex.prims:
        groups[raw_names[p.sid - 1] if p.sid else None].append(_poly(p))
    if groups.get(None):
        ax.add_collection(LineCollection(groups.pop(None), colors="#d0d0d0", linewidths=0.3))
    for name, segs in groups.items():
        ax.add_collection(LineCollection(segs, colors=PREVIEW_COLORS.get(name, "#000000"), linewidths=0.8, label=f"{name} ({len(segs)})"))
    ax.autoscale()
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xticks([]); ax.set_yticks([])
    FsPath(path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return path

In [ ]:
%%writefile -a indlabel.py


def find_drawings(root):
    root = FsPath(root)
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in (".dxf", ".dwg"))


def source_group(path, root):
    rel = FsPath(path).relative_to(root)
    return rel.parts[0] if len(rel.parts) > 1 else rel.stem


def convert_dwgs(src_root, dxf_root, oda_path="ODAFileConverter", version="ACAD2018"):
    src_root, dxf_root = FsPath(src_root), FsPath(dxf_root)
    dwgs = [p for p in find_drawings(src_root) if p.suffix.lower() == ".dwg"]
    if not dwgs:
        return {"dwg": 0, "converted": 0}
    exe = shutil.which(oda_path) or (oda_path if FsPath(oda_path).exists() else None)
    if exe is None:
        print("ODA File Converter not found: convert DWGs locally, or install it (see Stage 1). Skipping", len(dwgs), "DWG files.")
        return {"dwg": len(dwgs), "converted": 0}
    cmd = [exe, str(src_root), str(dxf_root), version, "DXF", "1", "1", "*.DWG"]
    if shutil.which("xvfb-run"):
        cmd = ["xvfb-run", "-a"] + cmd
    subprocess.run(cmd, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    done = sum(1 for d in dwgs if (dxf_root / d.relative_to(src_root)).with_suffix(".dxf").exists())
    return {"dwg": len(dwgs), "converted": done}



def process_file(path, group, rules, raw_names, svg_dir, preview_dir):
    stem = f"{group}__{FsPath(path).stem}"
    rep = {"file": str(path), "group": group, "id": stem, "flags": []}
    try:
        doc, ex, status = explode_file(path, rules)
        rep["read_status"] = status
        units = infer_units(doc, ex.prims)
        scale_to_mm(ex, units["mm_factor"])
        assign_instances(ex)
        doors = label_prims(ex, rules, raw_names)
        geom = write_svg(ex, FsPath(svg_dir) / f"{stem}.svg")
        preview(ex, raw_names, FsPath(preview_dir) / f"{stem}.png", title=f"{stem}  ({units['mm_factor']} mm/unit, {units['status']})")
    except Exception as e:
        rep.update({"flags": ["error"], "error": repr(e)[:300]})
        return rep
    n = len(ex.prims)
    sid_counts = Counter(p.sid for p in ex.prims)
    inst = defaultdict(set)
    for p in ex.prims:
        if p.iid >= 0:
            inst[p.sid].add(p.iid)
    unl = Counter(p.layer for p in ex.prims if p.sid == 0)
    widths = [d["width_mm"] for d in doors]
    rep.update({
        "svg": str(FsPath(svg_dir) / f"{stem}.svg"), "preview": str(FsPath(preview_dir) / f"{stem}.png"),
        "n_prims": n, "n_text": len(ex.texts), "labelled_frac": round(1 - sid_counts.get(0, 0) / max(n, 1), 4),
        "mm_factor": units["mm_factor"], "unit_status": units["status"], "insunits": units["insunits"],
        "header_overridden": units["header_overridden"], "door_leaf_mm": units["door_leaf_mm"],
        "wall_thickness_mm": units["wall_thickness_mm"],
        "n_doors": len(doors), "door_width_median_mm": float(np.median(widths)) if widths else None,
        "door_subtypes": dict(Counter(d["subtype"] for d in doors)),
        "class_prims": {raw_names[k - 1]: v for k, v in sid_counts.items() if k},
        "class_instances": {raw_names[k - 1]: len(v) for k, v in inst.items()},
        "unlabelled_layers": dict(unl.most_common(25)), "explode_errors": dict(ex.errors),
        "implausible_instances": getattr(ex, "implausible", {}),
        "size_mm": geom["size_mm"],
    })
    wall_prims = sum(v for k, v in rep["class_prims"].items() if k == "wall")
    if wall_prims < 20:
        rep["flags"].append("no_walls")
    if not doors:
        rep["flags"].append("no_doors")
    elif not 600 <= rep["door_width_median_mm"] <= 1400:
        rep["flags"].append("door_size_off")
    if units["status"] == "unverified":
        rep["flags"].append("units_unverified")
    if rep["labelled_frac"] < 0.03:
        rep["flags"].append("low_coverage")
    return rep


def _process_job(args):
    return process_file(*args)


def process_all(items, rules, raw_names, svg_dir, preview_dir, workers=2):
    jobs = [(str(p), g, rules, raw_names, str(svg_dir), str(preview_dir)) for p, g in items]
    t0 = time.time()
    if workers <= 1:
        reps = [_process_job(j) for j in jobs]
    else:
        from multiprocessing import Pool
        with Pool(workers) as pool:
            reps = list(pool.imap_unordered(_process_job, jobs, chunksize=1))
    print(f"processed {len(jobs)} drawings in {time.time() - t0:.0f}s")
    return pd.DataFrame(reps).sort_values("id").reset_index(drop=True)


def unmatched_layers(report, top=40):
    tot, files = Counter(), Counter()
    for d in report["unlabelled_layers"].dropna():
        tot.update(d)
        files.update(d.keys())
    return pd.DataFrame([{"layer": k, "unlabelled_prims": v, "files": files[k]} for k, v in tot.most_common(top)])

In [ ]:
%%writefile -a indlabel.py


def split_floors(dxf_dir, plan, log=print):
    dxf_dir = FsPath(dxf_dir)
    jobs = [(role, tower, block, tag) for role, floors in plan.items() for tower, block, tag in floors]
    for role, tower, block, tag in jobs:
        dest = dxf_dir / role / f"{tower}_{tag}.dxf"
        if dest.exists():
            continue
        src = next((dxf_dir / tower).glob("*.dxf"))
        t0 = time.time()
        doc, _ = recover.readfile(str(src))
        msp = doc.modelspace()
        kept = 0
        for e in list(msp):
            if e.dxftype() == "INSERT" and e.dxf.name == block:
                kept += 1
            else:
                msp.delete_entity(e)
        dest.parent.mkdir(parents=True, exist_ok=True)
        doc.saveas(str(dest))
        log(f"{role}/{dest.name}: kept {kept} insert, {dest.stat().st_size / 1e6:.1f} MB, {time.time() - t0:.0f}s")
    masters = dxf_dir.parent / "dxf_master"
    for tower in {t for _, t, _, _ in jobs}:
        if (dxf_dir / tower).exists():
            masters.mkdir(exist_ok=True)
            shutil.move(str(dxf_dir / tower), str(masters / tower))

In [ ]:
%%writefile -a indlabel.py


def assign_split_groups(report, test_groups=(), val_groups=(), val_frac=0.15, test_frac=0.15, seed=0):
    groups = report.groupby("group").size().to_dict()
    split = {g: "test" for g in test_groups if g in groups}
    split.update({g: "val" for g in val_groups if g in groups})
    free = sorted(g for g in groups if g not in split)
    rnd = np.random.default_rng(seed)
    rnd.shuffle(free)
    total = sum(groups.values())
    need = {"test": test_frac * total - sum(groups[g] for g, s in split.items() if s == "test"),
            "val": val_frac * total - sum(groups[g] for g, s in split.items() if s == "val")}
    for g in free:
        left = [x for x in free if x not in split]
        if len(left) <= 1 and "train" not in split.values():
            target = "train"
        elif need["test"] > 0:
            target = "test"
        elif need["val"] > 0:
            target = "val"
        else:
            target = "train"
        split[g] = target
        if target != "train":
            need[target] -= groups[g]
    if len(groups) < 3:
        print("WARNING: fewer than 3 source groups; splits cannot be source-disjoint")
    return split


def _graph_job(args):
    svg, cfg_dict, lut, stuff, min_wall, min_frac, wall_id, other_id, fpgraph_dir = args
    import sys
    if fpgraph_dir not in sys.path:
        sys.path.insert(0, fpgraph_dir)
    import fpgraph
    try:
        cfg = fpgraph.GraphConfig(**cfg_dict)
        P = fpgraph.split_long_lines(fpgraph.parse_svg(svg, cfg.mm_per_unit), cfg.max_prim_len_mm)
        y_all = lut[P["raw"]].astype(np.int64)
        windows = fpgraph.tile_windows(fpgraph.midpoints(P), cfg)
        kept = []
        for t, (win, idx) in enumerate(windows):
            y = y_all[idx]
            if int((y == wall_id).sum()) < min_wall or float((y != other_id).mean()) < min_frac:
                continue
            g = fpgraph.build_graph(P, idx, win, cfg, lut, stuff)
            g["meta"] = {"drawing": FsPath(svg).stem, "tile": t, "window": [round(v, 1) for v in win],
                         "n_nodes": int(len(idx)), "n_edges": int(g["edge_index"].shape[1]), "n_text": int(P["n_text"])}
            kept.append(g)
        return svg, kept, len(windows), ""
    except Exception as e:
        return svg, [], 0, repr(e)


def build_split(svgs, split, out_dir, fpgraph, cfg, lut, stuff, targets, min_wall_nodes=20, min_labelled_frac=0.03,
                workers=2, tiles_per_shard=256):
    writer = fpgraph.ShardWriter(out_dir, split, tiles_per_shard)
    fp_dir = str(FsPath(fpgraph.__file__).resolve().parent)
    jobs = [(str(s), asdict(cfg), lut, stuff, min_wall_nodes, min_labelled_frac, targets.index("wall"), targets.index("other"), fp_dir)
            for s in svgs]
    if workers <= 1:
        results = map(_graph_job, jobs)
        pool = None
    else:
        from multiprocessing import Pool
        pool = Pool(workers)
        results = pool.imap_unordered(_graph_job, jobs, chunksize=1)
    errors, dropped, t0 = [], 0, time.time()
    for svg, kept, n_all, err in results:
        if err:
            errors.append({"file": svg, "error": err})
        dropped += n_all - len(kept)
        for t in kept:
            writer.add(t)
    if pool is not None:
        pool.close()
        pool.join()
    writer.flush()
    index = pd.DataFrame(writer.index)
    index.to_parquet(FsPath(out_dir) / f"index_{split}.parquet", index=False)
    pd.DataFrame(errors, columns=["file", "error"]).to_csv(FsPath(out_dir) / f"errors_{split}.csv", index=False)
    print(f"{split}: {len(svgs)} drawings -> {len(index)} tiles kept, {dropped} dropped, {len(errors)} errors, {time.time() - t0:.0f}s")
    return index


def write_card(out_dir, info):
    rows = "\n".join(f"| {s['split']} | {s['drawings']} | {s['tiles']} | {s['nodes']:,} | {s['edges']:,} | {s['edge_pos_weight']} |"
                     for s in info["stats"])
    groups = "\n".join(f"| {g} | {s} | {n} |" for g, s, n in info["groups"])
    card = f"""# Indian CAD fine-tuning set (Phase B)

Auto-labelled Indian floor plans for fine-tuning the Phase A GNN. Same tile, graph and label format as the FloorPlanCAD
shards in this repository, built with the same `fpgraph.py` and `config.json`.

**Labels** come from layer and block-name rules (`label_rules.json`) plus geometry (leaf-arc count decides single, double
or sliding doors; door swings and wall thickness confirm the units). They are automatic, so treat them as silver
labels: every drawing was checked on a preview before inclusion (`review.csv`).

**Provenance.** Drawings come from the sources listed below. Raw drawings are not redistributed. Respect each source's
terms; this derivative is for non-commercial research.

## Splits (source-disjoint)

| split | drawings | tiles | nodes | edges | edge pos_weight |
|---|---|---|---|---|---|
{rows}

| source group | split | drawings |
|---|---|---|
{groups}

Tiles were kept only if they have at least {info['min_wall_nodes']} wall nodes and at least {info['min_labelled_frac']:.0%} labelled nodes, which drops
elevations, sections, title blocks and empty margins.
"""
    (FsPath(out_dir) / "README.md").write_text(card)


def push_folder(out_dir, repo_id, path_in_repo="indian", token=None, private=True):
    from huggingface_hub import HfApi, get_token
    token = token or get_token()
    if not token:
        print("No Hugging Face token found; set HF_TOKEN (Colab secret) and re-run.")
        return None
    api = HfApi(token=token)
    api.create_repo(repo_id, repo_type="dataset", private=private, exist_ok=True)
    out_dir = FsPath(out_dir)
    api.upload_folder(repo_id=repo_id, repo_type="dataset", folder_path=str(out_dir), path_in_repo=path_in_repo,
                      ignore_patterns=["graphs/**"], commit_message="Phase B metadata")
    for sub in sorted(p for p in (out_dir / "graphs").iterdir() if p.is_dir()):
        api.upload_folder(repo_id=repo_id, repo_type="dataset", folder_path=str(sub), path_in_repo=f"{path_in_repo}/graphs/{sub.name}",
                          commit_message=f"Phase B shards: {sub.name}")
        print("uploaded", sub.name)
    url = f"https://huggingface.co/datasets/{repo_id}/tree/main/{path_in_repo}"
    print("pushed to", url)
    return url

### 2c. The Phase E modules
New code: the exporter wrapper, the inference and merging, the adapter into the Track A core, the comparison and the one-call function.

### 2b. `cvexport.py`: DXF to the SVG the model reads
The contract from Phase A/B: inference SVGs come from **our own exporter** (`indlabel.write_svg`), so the model sees exactly the geometry it was trained on: mm coordinates, true arcs, y flipped, text kept as `<text>`, no layer names.

Two choices here matter:
- **Label-free SVG.** The exporter only writes `semantic-id` / `instance-id` for primitives that carry a label. We write the SVG *before* any labelling step runs, so no primitive has one. The model cannot peek at the CAD layer rules.
- **Same entities as Track A.** With `floor_only=True` only the floor block's entities are exploded (`pick_floor_insert`, the same call Stage 2a makes: the INSERT whose block has the most entities, or the one named in `floor_block`), so both tracks look at the same drawing and not at the key plans and schedules that sit beside it. A DWG that holds several floors as separate blocks needs `floor_block` to pick one; `list_floor_blocks` shows the candidates.

The exporter returns `origin_mm = [x0, y1]`: `svg_x = x - x0` and `svg_y = y1 - y`. We save it in a `.json` next to the SVG, because the adapter must turn model output back into the original drawing coordinates so that polygons, doors and walls land where Track A puts them.

In [ ]:
%%writefile cvexport.py
import json
from pathlib import Path

import indlabel
import tracka_core


def list_floor_blocks(doc, top=15):
    rows = [{"block": i.dxf.name, "entities": len(doc.blocks[i.dxf.name])} for i in doc.modelspace().query("INSERT")]
    return sorted(rows, key=lambda r: -r["entities"])[:top]


def floor_entities(doc, floor_only, floor_block=None):
    if floor_only:
        try:
            top = tracka_core.pick_floor_insert(doc, floor_block)
            return top, list(top.virtual_entities())
        except (ValueError, StopIteration):
            if floor_block:
                raise ValueError(f"no INSERT of block {floor_block!r} in modelspace; see list_floor_blocks")
    return None, list(doc.modelspace())


def export_svg(dxf_path, svg_path, floor_only=True, floor_block=None, mm_factor=None, rules=None, doc=None):
    rules = rules or indlabel.Rules()
    status = "preloaded"
    if doc is None:
        doc, status = indlabel.read_dxf(dxf_path)
    top, entities = floor_entities(doc, floor_only, floor_block)
    ex = indlabel.Exploded()
    indlabel.explode(entities, ex, rules.compiled())
    units = indlabel.infer_units(doc, ex.prims)
    factor = float(mm_factor or units["mm_factor"])
    indlabel.scale_to_mm(ex, factor)
    geom = indlabel.write_svg(ex, svg_path)
    info = {**geom, "units": units, "mm_factor": factor, "read_status": status, "n_prims": len(ex.prims),
            "n_text": len(ex.texts), "floor_block": top.dxf.name if top is not None else None,
            "source_file": Path(dxf_path).name, "explode_errors": dict(ex.errors)}
    Path(svg_path).with_suffix(".json").write_text(json.dumps(info, default=float, indent=1))
    return doc, top, ex, info


def load_info(svg_path):
    return json.loads(Path(svg_path).with_suffix(".json").read_text())

**Silver-label SVG (for the sanity check only).** After the clean SVG is written, the same exploded primitives are labelled with the Phase B rules and written again. Nothing from this file reaches the model. It is used in step 5 to measure how well the model agrees with the rule-based labels on this floor, and in step 6 as an *oracle* that feeds the adapter perfect labels, so the adapter's own errors can be told apart from the model's.

In [ ]:
%%writefile -a cvexport.py


def export_silver_svg(ex, svg_path, raw_names, rules=None):
    rules = rules or indlabel.Rules()
    indlabel.assign_instances(ex)
    indlabel.label_prims(ex, rules, raw_names)
    return indlabel.write_svg(ex, svg_path)

### 2c. `gnninfer.py`: SVG to per-primitive classes and instances
**Tiling.** `fpgraph.tile_windows` cuts a drawing into 10 m tiles with 1 m overlap (a quadtree split for dense areas), and `fpgraph.build_graph` builds the graph of each. Training code throws the tile's index list away; here it is **kept** (`t["idx"]`), because it says which global primitives each tile node is. Without it the tiles cannot be stitched back together.

**Merging.** A primitive in an overlap strip is seen by two (or four) tiles. Its 12 class logits are **averaged** over the tiles that contain it, then a softmax gives the class and its probability. The same is done for edges: each tile edge is keyed by its two *global* primitive ids, and the logits of duplicates are averaged. Instances are then decoded **once on the merged graph** (`gnneval.decode_instances`: threshold the edge probability, connected components over thing primitives, length-weighted majority class). Two tiles that both contain half of a door therefore end up with one door, because the shared primitives glue the components together.

**Confidence.** For every primitive: the softmax probability of the class it finally gets (for a door or window primitive that is the class voted by its instance). A drawing region that no tile covers (the quadtree caps a tile at 4,096 nodes, and tiles under 8 nodes are dropped) gets class `other` with probability 0, and the count of such primitives is reported.

In [ ]:
%%writefile gnninfer.py
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import fpgraph
from gnnmodel import FloorGAT, ModelConfig
from gnndata import collate, make_batches, to_device
from gnneval import decode_instances, panoptic_counts, pq_from_counts, f1_from_conf

CLASSES = fpgraph.TARGET_CLASSES
OTHER = CLASSES.index("other")


def load_model(ckpt_path, device):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = FloorGAT(ModelConfig(**ck["cfg"])).to(device)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, ck


def read_threshold(path, default=0.5):
    p = Path(path)
    return float(json.loads(p.read_text())["threshold"]) if p.exists() else default


def graph_config(phase_a, mm_per_unit=1.0):
    return fpgraph.GraphConfig(**{**phase_a["graph"], "mm_per_unit": mm_per_unit})


def read_primitives(svg_path, cfg):
    return fpgraph.split_long_lines(fpgraph.parse_svg(svg_path, cfg.mm_per_unit), cfg.max_prim_len_mm)


def build_tiles(P, cfg, lut, stuff, src=1):
    tiles = []
    for t, (win, idx) in enumerate(fpgraph.tile_windows(fpgraph.midpoints(P), cfg)):
        g = fpgraph.build_graph(P, idx, win, cfg, lut, stuff)
        g.update({"idx": idx.astype(np.int64), "window": win, "src": src})
        tiles.append(g)
    return tiles


@torch.no_grad()
def predict_tiles(model, tiles, device, node_budget=12000, max_tiles=8, amp=False):
    sizes = np.array([len(t["y"]) for t in tiles])
    outs = [None] * len(tiles)
    for group in make_batches(sizes, np.arange(len(tiles)), node_budget, max_tiles):
        batch = to_device(collate([tiles[i] for i in group]), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            out = model(batch["x"], batch["edge_index"], batch["edge_attr"])
        nl = out["node_logits"].float().cpu().numpy()
        el = out["edge_logits"].float().cpu().numpy()
        nptr, eptr = batch["node_ptr"].cpu().numpy(), batch["edge_ptr"].cpu().numpy()
        for b, i in enumerate(group):
            outs[i] = (nl[nptr[b]:nptr[b + 1]], el[eptr[b]:eptr[b + 1]])
    return outs


def merge_tiles(n_prims, tiles, outs):
    acc = np.zeros((n_prims, len(CLASSES)))
    cnt = np.zeros(n_prims)
    keys, vals = [], []
    for t, (nl, el) in zip(tiles, outs):
        idx = t["idx"]
        acc[idx] += nl
        cnt[idx] += 1
        ei = t["edge_index"].astype(np.int64)
        a, b = idx[ei[0]], idx[ei[1]]
        keys.append(np.minimum(a, b) * n_prims + np.maximum(a, b))
        vals.append(el.astype(np.float64))
    covered = cnt > 0
    logits = np.where(covered[:, None], acc / np.maximum(cnt, 1)[:, None], 0.0)
    if keys:
        key, val = np.concatenate(keys), np.concatenate(vals)
        uk, inv = np.unique(key, return_inverse=True)
        edge_logit = np.bincount(inv, weights=val) / np.bincount(inv)
        edge_index = np.stack([uk // n_prims, uk % n_prims])
    else:
        edge_logit, edge_index = np.zeros(0), np.zeros((2, 0), dtype=np.int64)
    return {"logits": logits, "covered": covered, "n_tiles_per_prim": cnt, "edge_index": edge_index, "edge_logit": edge_logit}


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def decode_scene(P, merged, thr):
    n = len(P["ptype"])
    prob_all = softmax(merged["logits"])
    covered = merged["covered"]
    pred = prob_all.argmax(1)
    pred[~covered] = OTHER
    edge_prob = 1.0 / (1.0 + np.exp(-merged["edge_logit"]))
    seg, seg_cls, seg_len = decode_instances(pred, edge_prob, merged["edge_index"], P["length"].astype(np.float64), thr)
    cls = pred.copy()
    thing = seg >= 0
    cls[thing] = seg_cls[seg[thing]]
    prob = prob_all[np.arange(n), cls]
    prob[~covered] = 0.0
    return {"pred": pred, "cls": cls, "prob": prob, "seg": seg, "seg_cls": seg_cls, "seg_len": seg_len,
            "covered": covered, "thr": float(thr)}

**Reading silver labels back.** `silver_arrays` turns a labelled SVG (parsed with the same `fpgraph` code) into the ground-truth arrays the Phase C/D evaluator uses: harmonised class per primitive and a dense instance id for thing primitives, built with the rule `build_graph` uses. `silver_agreement` then scores a decoded scene with the **same functions as training** (`f1_from_conf`, `panoptic_counts`, `pq_from_counts`), so the numbers are comparable with the notebook-8 results. `scene_from_silver` builds a scene out of the labels themselves, probability 1: the oracle.

In [ ]:
%%writefile -a gnninfer.py


def silver_arrays(P_gt, lut, stuff):
    y = lut[P_gt["raw"]].astype(np.int64)
    thing = ~stuff[y] & (y != OTHER) & (P_gt["inst"] >= 0)
    inst = np.full(len(y), -1, dtype=np.int64)
    if thing.any():
        key = P_gt["raw"].astype(np.int64)[thing] * 10_000_000 + P_gt["inst"].astype(np.int64)[thing]
        _, inv = np.unique(key, return_inverse=True)
        inst[thing] = inv
    return y, inst


def scene_from_silver(P_gt, lut, stuff):
    y, inst = silver_arrays(P_gt, lut, stuff)
    n_inst = int(inst.max()) + 1 if (inst >= 0).any() else 0
    seg_cls = np.zeros(n_inst, dtype=np.int64)
    seg_len = np.zeros(n_inst)
    has = inst >= 0
    seg_cls[inst[has]] = y[has]
    np.add.at(seg_len, inst[has], P_gt["length"].astype(np.float64)[has])
    return {"pred": y.copy(), "cls": y.copy(), "prob": np.ones(len(y)), "seg": inst, "seg_cls": seg_cls, "seg_len": seg_len,
            "covered": np.ones(len(y), dtype=bool), "thr": None}


def silver_agreement(P_gt, scene, lut, stuff):
    y, inst = silver_arrays(P_gt, lut, stuff)
    length = P_gt["length"].astype(np.float64)
    C = len(CLASSES)
    pred = scene["pred"]
    conf = np.bincount(y * C + pred, minlength=C * C).reshape(C, C)
    conf_len = np.bincount(y * C + pred, weights=length, minlength=C * C).reshape(C, C)
    counts = panoptic_counts(y, inst, length, scene["seg"], scene["seg_cls"], scene["seg_len"])
    pq = pq_from_counts(counts)
    f1c, sup, acc = f1_from_conf(conf)
    f1l, supl, _ = f1_from_conf(conf_len)
    table = pd.DataFrame({"class": CLASSES, "silver_prims": sup.astype(int), "f1": f1c.round(3), "f1_length_weighted": f1l.round(3),
                          "pq": pq["pq_c"].round(3), "silver_instances": (counts[:, 0] + counts[:, 2]).astype(int),
                          "matched": counts[:, 0].astype(int), "extra_pred": counts[:, 1].astype(int)})
    summary = {"accuracy": float(acc), "wF1_length": float((f1l * supl).sum() / max(supl.sum(), 1e-9)),
               "macro_f1_present": float(f1c[sup > 0].mean()), "pq": pq["pq"], "rq": pq["rq"], "sq": pq["sq"]}
    return table, summary

### 2d. `cvadapter.py`: model output to the Track A geometry core
**The idea.** The Track A core (walls to solid, holes to rooms, door sides, adjacency, corridors, manifest) never looks at the CAD file directly. It consumes (1) *leaves*: line pieces tagged with a layer, (2) *openings*, (3) *room tags* and (4) a scale record. The adapter produces exactly those four things from the GNN's output and then **runs the unchanged Track A `run()` and `build_manifest()`**. Only four names inside `tracka_core` are swapped for the duration of the call (`load_floor`, `build_openings`, `room_tags`, `check_dimensions`), and `patched` puts them back afterwards.

| Track A reads | CV track supplies |
|---|---|
| geometry on layer `A-WALL`, `A-CW`, `A-RAIL` (boundaries) | primitives predicted `wall`, `curtain_wall`, `railing` |
| geometry inside `A-WIN` blocks (glazing seals the opening) | primitives of accepted `window` instances |
| `D1-800` door blocks and their bounding boxes | accepted `door_*` instances and their oriented extent |
| `RD` blocks with `RT` (name) and `SZ` (designed size) | `<text>` items from the SVG; a size-looking text is attached to the nearest name |
| `DIMENSION` entities | none (the SVG has no dimension entities), so `dimensions = []` |

**Leaves.** Every kept primitive becomes a polyline in drawing coordinates: lines keep their two ends, arcs are re-flattened from centre, radius and sweep at 5 mm chord tolerance (the Track A value), circles and ellipses become 16-gons.

**Re-joining split lines.** The graph builder cuts every line longer than 2 m into pieces of at most 2 m, so a 8 m wall face reaches the adapter as four primitives. Handed to Track A as they are, these would give four times the wall records and plant false jamb points every 2 m. `merge_touching_lines` undoes the split: lines with the same direction (to 0.006 degrees) and the same offset (to 1 mm) are sorted along their axis and joined while they touch (gap at most 1 mm). Only touching pieces are joined, so real gaps in a wall stay gaps.

In [ ]:
%%writefile cvadapter.py
import re
import math
import time
import contextlib
import html
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import shapely
from shapely import STRtree
from shapely.geometry import Polygon, LineString, box
from scipy.spatial.distance import cdist

import tracka_core as core
from fpgraph import TARGET_CLASSES

CV_CONFIG = {
    "instance_size_mm": {"door": (450.0, 6000.0), "window": (300.0, 10000.0)},
    "axis_search_pad": 300.0,
    "axis_bin_deg": 2.0,
    "axis_tol_deg": 3.0,
    "arc_flatten_tol": 5.0,
    "circle_segments": 16,
    "merge_tol": 1.0,
    "swing_span_rad": (math.radians(20.0), math.radians(180.0) + 0.01),
    "min_extent": 10.0,
    "size_attach_radius": 1500.0,
    "room_text_max_len": 40,
    "min_room_letters": 3,
    "door_code_regex": r"^[A-Z]{1,3}[- ]?\d+(\s*[X*]\s*\d+)?[A-Z]?(-\d+)?$",
    "boundary_conf_pad": 30.0,
    "wall_conf_pad": 20.0,
}

BOUNDARY_LAYER = {"wall": "A-WALL", "curtain_wall": "A-CW", "railing": "A-RAIL"}
DOOR_CLASSES = {c for c in TARGET_CLASSES if c.startswith("door")}
OPENING_CLASSES = DOOR_CLASSES | {"window"}
DXF_TYPE = {0: "LINE", 1: "ARC", 2: "CIRCLE", 3: "ELLIPSE", 4: "SPLINE"}


@contextlib.contextmanager
def patched(**overrides):
    saved = {k: getattr(core, k) for k in overrides}
    for k, v in overrides.items():
        setattr(core, k, v)
    try:
        yield
    finally:
        for k, v in saved.items():
            setattr(core, k, v)


def to_wcs(P, origin):
    x0, y1 = origin
    pts = P["pts"].astype(np.float64)
    pts = np.stack([pts[..., 0] + x0, y1 - pts[..., 1]], -1)
    ctr = P["center"].astype(np.float64)
    ctr = np.stack([ctr[:, 0] + x0, y1 - ctr[:, 1]], 1)
    return {"pts": pts, "center": ctr, "ptype": P["ptype"], "radius": P["radius"].astype(np.float64),
            "sweep": np.abs(P["sweep"]).astype(np.float64), "length": P["length"].astype(np.float64)}


def prim_polyline(Pw, i, cv):
    t, p = int(Pw["ptype"][i]), Pw["pts"][i]
    if t == 0:
        return [tuple(p[0]), tuple(p[3])], False
    c, r = Pw["center"][i], Pw["radius"][i]
    if t in (2, 3):
        rx, ry = abs(p[0][0] - c[0]), abs(p[1][1] - c[1])
        a = np.linspace(0, 2 * math.pi, cv["circle_segments"], endpoint=False)
        return [(float(c[0] + rx * math.cos(k)), float(c[1] + ry * math.sin(k))) for k in a], True
    span = Pw["sweep"][i]
    if t == 1 and r > 0 and span > 0 and np.isfinite(c).all():
        a0 = math.atan2(p[0][1] - c[1], p[0][0] - c[0])
        a1 = math.atan2(p[1][1] - c[1], p[1][0] - c[0])
        sign = 1.0 if (a1 - a0 + math.pi) % (2 * math.pi) - math.pi >= 0 else -1.0
        step = 2 * math.acos(max(-1.0, 1.0 - cv["arc_flatten_tol"] / r))
        n = max(2, int(math.ceil(span / max(step, 1e-3))))
        pts = [(float(c[0] + r * math.cos(a0 + sign * span * k / n)), float(c[1] + r * math.sin(a0 + sign * span * k / n)))
               for k in range(n + 1)]
        pts[0], pts[-1] = tuple(p[0]), tuple(p[3])
        return pts, False
    return [tuple(q) for q in p], False

**Instances.** The decoded scene gives a segment id per thing primitive. An instance whose class is a door or window is kept only if its overall size is plausible (door 450 mm to 6 m, window 300 mm to 10 m: the Phase B size guard). Smaller fragments, such as one stray arc predicted as a door, are dropped and contribute nothing. The instance confidence is the **minimum** softmax probability over its primitives.

In [ ]:
%%writefile -a cvadapter.py


def build_instances(Pw, scene, cv):
    cls, prob, seg = scene["cls"], scene["prob"], scene["seg"]
    idx = np.where(seg >= 0)[0]
    if not len(idx):
        return []
    order = idx[np.argsort(seg[idx], kind="stable")]
    groups = np.split(order, np.flatnonzero(np.diff(seg[order])) + 1)
    out = []
    for members in groups:
        name = TARGET_CLASSES[int(cls[members[0]])]
        if name not in OPENING_CLASSES:
            continue
        kind = "door" if name in DOOR_CLASSES else "window"
        size = float(np.ptp(Pw["pts"][members].reshape(-1, 2), axis=0).max())
        lo, hi = cv["instance_size_mm"][kind]
        out.append({"id": int(seg[members[0]]), "cls": name, "kind": kind, "members": members, "size_mm": size,
                    "accepted": lo <= size <= hi, "conf": float(prob[members].min())})
    return out


def merge_touching_lines(ends, tol=1.0):
    n = len(ends)
    if n < 2:
        return ends
    d = ends[:, 1] - ends[:, 0]
    L = np.hypot(d[:, 0], d[:, 1])
    u = d / np.maximum(L, 1e-12)[:, None]
    flip = (u[:, 0] < -1e-9) | ((np.abs(u[:, 0]) <= 1e-9) & (u[:, 1] < 0))
    u = np.where(flip[:, None], -u, u)
    a = np.where(flip[:, None], ends[:, 1], ends[:, 0])
    t0 = (a * u).sum(1)
    t1 = t0 + L
    off = u[:, 0] * a[:, 1] - u[:, 1] * a[:, 0]
    kt = np.round(np.arctan2(u[:, 1], u[:, 0]) / 1e-4).astype(np.int64)
    ko = np.round(off).astype(np.int64)
    order = np.lexsort((t0, ko, kt))
    kt_s, ko_s, t0_s, t1_s = (v[order].tolist() for v in (kt, ko, t0, t1))
    firsts, tops = [], []
    for k in range(n):
        if k and kt_s[k] == kt_s[k - 1] and ko_s[k] == ko_s[k - 1] and t0_s[k] <= tops[-1] + tol:
            tops[-1] = max(tops[-1], t1_s[k])
        else:
            firsts.append(k)
            tops.append(t1_s[k])
    f = order[firsts]
    start = a[f]
    stop = start + u[f] * (np.array(tops) - t0[f])[:, None]
    return np.stack([start, stop], 1)


def cv_leaves(Pw, scene, instances, cv):
    cls, ptype = scene["cls"], Pw["ptype"]
    keep_window = np.zeros(len(ptype), dtype=bool)
    for ins in instances:
        if ins["accepted"] and ins["kind"] == "window":
            keep_window[ins["members"]] = True
    leaves, boundary = [], []
    for name, layer in BOUNDARY_LAYER.items():
        k = TARGET_CLASSES.index(name)
        sel = np.where(cls == k)[0]
        boundary.append(sel)
        lines = sel[ptype[sel] == 0]
        for a, b in merge_touching_lines(Pw["pts"][lines][:, [0, 3]], cv["merge_tol"]):
            leaves.append(core.Leaf("LINE", layer, "", name, [tuple(a), tuple(b)], False, 0.0))
        for i in sel[ptype[sel] != 0]:
            pts, closed = prim_polyline(Pw, i, cv)
            leaves.append(core.Leaf(DXF_TYPE[int(ptype[i])], layer, "", name, pts, closed, float(Pw["radius"][i])))
    for i in np.where((cls == TARGET_CLASSES.index("window")) & keep_window)[0]:
        pts, closed = prim_polyline(Pw, i, cv)
        leaves.append(core.Leaf(DXF_TYPE[int(ptype[i])], "A-WIN", "A-WIN", "window", pts, closed, float(Pw["radius"][i])))
    return leaves, np.concatenate(boundary).astype(np.int64)

### Openings from instances
Track A finds a door's footprint from its **block definition** and the insert matrix. A predicted door has no block, so the footprint is rebuilt from the instance's primitives, and everything after that is the same arithmetic as Track A's `build_openings`.

1. **Wall axis `u`.** Take the boundary segments near the instance (300 mm pad), build a length-weighted histogram of their angles in 2 degree bins, and use the peak (refined by the weighted mean within 3 degrees). A door leaf stands perpendicular to the wall and the swing arc is round, so only the *walls* can say which way the opening runs. `v` is the normal.
2. **Oriented box.** Project the instance's points on `u` and `v`; the min and max give a box whose `u` extent is the block width (`width_block_mm`) and whose `v` extent is the swing depth, like a block bounding box.
3. **Jambs, clear width, bridges.** Identical to Track A: boundary endpoints inside the box padded by 60 mm are jamb points, split in two groups along `u`, clear width = gap between the groups, one bridge per wall face. If the jambs are missing, the bridge is laid along `u` at the offset of the nearest parallel wall line.
4. **Width fallbacks.** Jamb clear opening, then swing-arc radius (doubled when two big arcs span the box, or when the model says `door_double`), then the box extent. There is no block name, so Track A's `block_name` fallback does not exist here.

Trace: a 900 mm door in a horizontal wall. Wall segments near it are almost all at 0 degrees, so `u = (1, 0)`. The instance's points span x from 5000 to 5900 and y from 0 to 900 (hinge, leaf, arc). Wall endpoints at x = 5000 and x = 5900 on both wall faces fall inside the padded box, the split at the mid-point gives left 5000 and right 5900, so clear width = 900 and two bridges close the gap.

In [ ]:
%%writefile -a cvadapter.py


def wall_axis(M, bnd_segs, seg_tree, cv):
    pad = cv["axis_search_pad"]
    lo, hi = M.min(0) - pad, M.max(0) + pad
    hit = seg_tree.query(box(lo[0], lo[1], hi[0], hi[1]), predicate="intersects")
    if not len(hit):
        return None, None
    s = bnd_segs[hit]
    d = s[:, 2:] - s[:, :2]
    L = np.hypot(d[:, 0], d[:, 1])
    ok = L > 1e-6
    s, d, L = s[ok], d[ok], L[ok]
    if not len(L):
        return None, None
    ang = np.arctan2(d[:, 1], d[:, 0]) % math.pi
    nb = int(round(180.0 / cv["axis_bin_deg"]))
    bins = np.round(np.degrees(ang) / cv["axis_bin_deg"]).astype(int) % nb
    mode = math.radians(int(np.bincount(bins, weights=L, minlength=nb).argmax()) * cv["axis_bin_deg"])
    dev = (ang - mode + math.pi / 2) % math.pi - math.pi / 2
    near = np.abs(dev) <= math.radians(cv["axis_tol_deg"])
    theta = mode + float((L[near] * dev[near]).sum() / L[near].sum())
    return theta, s[near]


def wall_offset(par, v, v0, v1):
    mid = (v0 + v1) / 2
    if par is None or not len(par):
        return mid
    off = ((par[:, :2] + par[:, 2:]) / 2) @ v
    dist = np.maximum(np.maximum(v0 - off, off - v1), 0.0) + 1e-6 * np.abs(off - mid)
    return float(off[int(dist.argmin())])


def opening_from_instance(ins, Pw, bnd_segs, endpoints, ep_tree, seg_tree, cfg, cv):
    mem = ins["members"]
    M = Pw["pts"][mem].reshape(-1, 2)
    theta, par = wall_axis(M, bnd_segs, seg_tree, cv)
    if theta is None:
        return None
    u = np.array([math.cos(theta), math.sin(theta)])
    v = np.array([-u[1], u[0]])
    pu, pv = M @ u, M @ v
    u0, u1, v0, v1 = float(pu.min()), float(pu.max()), float(pv.min()), float(pv.max())
    if u1 - u0 < cv["min_extent"]:
        u0, u1 = (u0 + u1) / 2 - cv["min_extent"] / 2, (u0 + u1) / 2 + cv["min_extent"] / 2
    if v1 - v0 < cv["min_extent"]:
        v0, v1 = (v0 + v1) / 2 - cv["min_extent"] / 2, (v0 + v1) / 2 + cv["min_extent"] / 2
    corners = [tuple(a * u + b * v) for a, b in ((u0, v0), (u1, v0), (u1, v1), (u0, v1))]
    obb = Polygon(corners)
    pad = obb.buffer(cfg["opening_search_pad"], join_style=2)
    idx = ep_tree.query(pad, predicate="contains")
    jambs = np.unique(np.round(endpoints[idx], 1), axis=0) if len(idx) else np.zeros((0, 2))
    arcs = (Pw["ptype"][mem] == 1) & (Pw["sweep"][mem] >= cv["swing_span_rad"][0]) & (Pw["sweep"][mem] <= cv["swing_span_rad"][1])
    radii = sorted(round(float(r), 1) for r in Pw["radius"][mem][arcs])
    block_w = u1 - u0
    leaf_w = max(radii) if radii else None
    big = [r for r in radii if r >= 0.8 * max(radii)] if radii else []
    geometric_double = len(big) >= 2 and abs(2 * big[-1] - block_w) <= 0.15 * block_w
    if geometric_double:
        leaf_w = 2 * big[-1]
    double_leaf = geometric_double or ins["cls"] == "door_double"
    clear_w, plug, centre = None, None, None
    if len(jambs) >= 2:
        proj, off = jambs @ u, jambs @ v
        split = (proj.max() + proj.min()) / 2
        left, right = proj <= split, proj > split
        if left.any() and right.any():
            clear_w = float(proj[right].min() - proj[left].max())
            bridges = []
            for li in np.where(left)[0]:
                ri = np.where(right)[0][np.argmin(np.abs(off[right] - off[li]))]
                bridges.append(LineString([tuple(jambs[li]), tuple(jambs[ri])]))
            if clear_w > 0.4 * block_w:
                plug = shapely.unary_union(bridges).buffer(cfg["snap_tol"], cap_style="flat")
                centre = shapely.MultiPoint([tuple(p) for p in jambs]).centroid
    ref_w = leaf_w or block_w
    if clear_w is not None and not (0.5 * ref_w <= clear_w <= 1.6 * ref_w):
        clear_w = None
    if plug is None:
        vw = wall_offset(par, v, v0, v1)
        edge = LineString([tuple(u0 * u + vw * v), tuple(u1 * u + vw * v)])
        plug = edge.buffer(cfg["snap_tol"], cap_style=2)
        centre = edge.interpolate(0.5, normalized=True)
    width = clear_w or leaf_w or block_w
    method = "jamb_clear_opening" if clear_w else ("leaf_arc_radius" if leaf_w else "block_extent")
    return {
        "kind": ins["kind"], "block": ins["cls"], "layer": None, "subtype": ins["cls"],
        "position": tuple(map(float, centre.coords[0])), "insert": tuple(map(float, centre.coords[0])),
        "rotation": float(math.degrees(theta) % 360.0), "mirrored": False, "axis": u.tolist(), "normal": v.tolist(),
        "width_mm": width, "width_method": method, "width_clear_mm": clear_w, "width_leaf_mm": leaf_w,
        "width_block_mm": block_w, "width_nominal_mm": None, "height_nominal_mm": None, "double_leaf": bool(double_leaf),
        "jamb_points": len(jambs), "_plug": plug, "_obb": obb, "cv_class": ins["cls"], "cv_confidence": ins["conf"],
        "cv_instance": ins["id"],
    }


def build_openings_cv(instances, Pw, bnd_segs, cfg, cv):
    if not len(bnd_segs):
        return []
    endpoints = np.vstack([bnd_segs[:, :2], bnd_segs[:, 2:]])
    ep_tree = STRtree(shapely.points(endpoints))
    seg_tree = STRtree(shapely.linestrings(bnd_segs.reshape(-1, 2, 2)))
    out = []
    for ins in instances:
        if ins["accepted"]:
            rec = opening_from_instance(ins, Pw, bnd_segs, endpoints, ep_tree, seg_tree, cfg, cv)
            if rec:
                out.append(rec)
    return out

### Room tags from the SVG text
Track A reads `RT` and `SZ` from `RD` blocks on layer `A-RD`. The SVG has no layers, only `<text>` items at drawing coordinates, so the same two ideas are rebuilt from text content:
- **Size text** (`3650X4170`, `1650 WIDE`): recognised by Track A's own `parse_size`. It is *not* a name; it is attached to the nearest name within 1.5 m, nearest pairs first, each size used once. That recovers `raw_size` and `nominal`, so the `size_check` cross-check still runs.
- **Name text**: at least 3 letters, at most 40 characters, and not a door or window code such as `D1`, `MD-1050` or `CW-1800X1200`. Texts matching Track A's `annotation_label_regex` (`UP`, `CUT-OUT`, ...) are flagged as annotations, exactly as Track A does.

This is a heuristic, because a plain `<text>` carries no layer. Titles, grid names and notes that pass the filter appear as labels. Those that fall outside every room end up in `unassigned_labels` (the classical manifest also has 35 of those), and those that fall inside a room would add a wrong label. Step 8 measures both.

In [ ]:
%%writefile -a cvadapter.py


def read_svg_texts(svg_path, origin):
    x0, y1 = origin
    raw = Path(svg_path).read_text(encoding="utf-8", errors="replace")
    out = []
    for m in re.finditer(r'<text x="([-\d.]+)" y="([-\d.]+)"[^>]*>(.*?)</text>', raw, re.S):
        s = html.unescape(m.group(3)).strip()
        if s:
            out.append({"text": s, "layer": "", "owner_layer": "", "insert": (float(m.group(1)) + x0, y1 - float(m.group(2)))})
    return out


def cv_tags(texts, cfg, cv):
    ann = re.compile(cfg["annotation_label_regex"], re.I)
    code = re.compile(cv["door_code_regex"])
    labels, sizes = [], []
    for t in texts:
        s = " ".join(t["text"].split())
        if not s:
            continue
        size = core.parse_size(s)
        if size is not None:
            sizes.append((t["insert"], s, size))
            continue
        if sum(ch.isalpha() for ch in s) < cv["min_room_letters"] or len(s) > cv["room_text_max_len"] or code.match(s.upper()):
            continue
        labels.append({"raw_label": s, "raw_size": "", "nominal": None, "position": t["insert"], "annotation": bool(ann.search(s))})
    real = [k for k, lab in enumerate(labels) if not lab["annotation"]]
    if real and sizes:
        D = cdist(np.array([labels[k]["position"] for k in real]), np.array([s[0] for s in sizes]))
        pairs = sorted(zip(*np.where(D <= cv["size_attach_radius"])), key=lambda ij: D[ij])
        used_l, used_s = set(), set()
        for i, j in pairs:
            if i in used_l or j in used_s:
                continue
            used_l.add(i)
            used_s.add(j)
            labels[real[i]]["raw_size"], labels[real[i]]["nominal"] = sizes[j][1], sizes[j][2]
    return labels

### Running the core, then CV provenance and confidence
`run_cv` ties it together. After `core.run` and `core.build_manifest` have produced the manifest with the Track A code, three things are changed on the result, and only these:
- `source` becomes `"cv_model"` on rooms, openings and walls.
- `confidence` becomes `min(Track A confidence, model confidence)`. For an opening the model confidence is its instance's minimum softmax probability. For a wall segment it is the minimum over the wall primitives that touch its rectangle. For a room it is the minimum over the boundary primitives (wall, curtain wall, railing) within 30 mm of its outline. `cv_*` fields carry the model-only numbers.
- `summary["cv"]` records how the manifest was produced (threshold, class counts, uncovered primitives, instances rejected).

The Pydantic model is validated again at the end, so the output provably has the same schema as the classical manifest.

In [ ]:
%%writefile -a cvadapter.py


def cv_scale(info):
    return {"insunits": 4, "insunits_source": "svg_export", "unit_name": "mm", "to_metre": 0.001, "floor_insert_scale": [1.0, 1.0],
            "block_units": 0, "dimlfac_header": 1.0, "issues": [], "cross_checks": {},
            "normalised_to_mm_factor": float(info["mm_factor"]), "unit_inference": info["units"]}


def min_prob_in(geoms, tree, probs, pad):
    out = []
    for g in geoms:
        hit = tree.query(g.buffer(pad), predicate="intersects")
        out.append((float(probs[hit].min()), float(probs[hit].mean())) if len(hit) else (None, None))
    return out


def run_cv(svg_path, info, Pw, scene, top=None, cfg=None, cv=None):
    cfg = cfg or core.CONFIG
    cv = {**CV_CONFIG, **(cv or {})}
    t0 = time.time()
    texts = read_svg_texts(svg_path, info["origin_mm"])
    tags = cv_tags(texts, cfg, cv)
    instances = build_instances(Pw, scene, cv)
    leaves, boundary_idx = cv_leaves(Pw, scene, instances, cv)
    ex = core.Exploded(leaves=leaves, texts=texts)
    scale = cv_scale(info)
    floor = top if top is not None else SimpleNamespace(dxf=SimpleNamespace(name=info.get("floor_block") or Path(svg_path).stem, insert=(0.0, 0.0, 0.0)))
    with patched(load_floor=lambda p, c: (None, floor, ex, scale),
                 build_openings=lambda doc, e, segs, c: build_openings_cv(instances, Pw, segs, c, cv),
                 room_tags=lambda e, c: tags,
                 check_dimensions=lambda e, s, c: []):
        r = core.run(svg_path, cfg)
    manifest = core.build_manifest(info.get("source_file") or Path(svg_path).name, r, cfg)
    annotate(manifest, r, Pw, scene, instances, boundary_idx, tags, cv)
    manifest["summary"]["elapsed_s"] = round(r["elapsed"], 1)
    manifest["summary"]["cv"]["adapter_elapsed_s"] = round(time.time() - t0, 1)
    core.ManifestM.model_validate(manifest)
    return manifest, r


def annotate(manifest, r, Pw, scene, instances, boundary_idx, tags, cv):
    prob = scene["prob"]
    b_probs = prob[boundary_idx]
    b_tree = STRtree(shapely.linestrings(Pw["pts"][boundary_idx])) if len(boundary_idx) else None
    geoms = {s["id"]: s["geom"] for s in r["spaces"]}
    rooms = manifest["rooms"]
    conf = min_prob_in([geoms[x["id"]] for x in rooms], b_tree, b_probs, cv["boundary_conf_pad"]) if b_tree is not None else [(None, None)] * len(rooms)
    for rm, (lo, mean) in zip(rooms, conf):
        rm["source"] = "cv_model"
        rm["cv_boundary_conf_min"], rm["cv_boundary_conf_mean"] = lo, mean
        if lo is not None:
            rm["confidence"] = round(min(rm["confidence"], lo), 4)
    walls = manifest["walls"]
    boxes = [LineString([w["start"], w["end"]]).buffer(w["thickness_mm"] / 2 + cv["wall_conf_pad"], cap_style=2) for w in walls]
    wconf = min_prob_in(boxes, b_tree, b_probs, 0.0) if b_tree is not None else [(None, None)] * len(walls)
    for w, (lo, mean) in zip(walls, wconf):
        w["source"] = "cv_model"
        w["cv_conf_min"], w["cv_conf_mean"] = lo, mean
        w["confidence"] = round(lo if lo is not None else 0.5, 4)
    by_id = {o["id"]: o for o in r["openings"]}
    for o in manifest["openings"]:
        src = by_id[o["id"]]
        o["source"] = "cv_model"
        o["cv_class"], o["cv_confidence"], o["cv_instance"] = src["cv_class"], round(src["cv_confidence"], 4), src["cv_instance"]
        o["confidence"] = round(min(o["confidence"], src["cv_confidence"]), 4)
    names = np.array(TARGET_CLASSES)
    symbols = []
    for s in range(len(scene["seg_cls"])):
        if names[scene["seg_cls"][s]] in ("stairs", "elevator"):
            m = np.where(scene["seg"] == s)[0]
            xy = Pw["pts"][m].reshape(-1, 2)
            symbols.append({"class": str(names[scene["seg_cls"][s]]), "bbox": [round(float(v), 1) for v in (*xy.min(0), *xy.max(0))],
                            "confidence": round(float(prob[m].min()), 4), "n_prims": int(len(m))})
    manifest["summary"]["cv"] = {
        "head2_threshold": scene["thr"],
        "primitives": int(len(prob)),
        "uncovered_primitives": int((~scene["covered"]).sum()),
        "class_counts": {n: int(c) for n, c in zip(TARGET_CLASSES, np.bincount(scene["cls"], minlength=len(TARGET_CLASSES)))},
        "opening_instances": {"accepted": sum(i["accepted"] for i in instances), "rejected_size": sum(not i["accepted"] for i in instances)},
        "openings_in_plan": len(r["openings"]),
        "text_items": len(tags),
        "symbols": symbols,
    }

### 2e. `cvcompare.py`: CV manifest against the classical manifest
A first, small version of the Phase F comparison on the same floor, so the first run already says how far the CV track is from Track A. The classical manifest is a **reference, not ground truth**: it has its own known limits (see Stage 2a), and where the two disagree, either can be wrong.

- **Schema.** Same top-level keys in the same order, and the Track A Pydantic model accepts both.
- **Rooms.** Hungarian matching on polygon IoU (maximise the total), pairs with IoU of at least 0.5 count as matched. Reported: matched count, mean IoU, median absolute area error, label agreement (exact after upper-casing, or at least half the words shared).
- **Openings.** Per kind, Hungarian matching on centre distance, gated at 600 mm. Reported: matched, precision, recall, and the mean absolute width error on matched pairs.
- **Adjacency.** Edges are compared only between rooms that matched on both sides (CV room mapped to its reference room), so an unmatched room does not count twice. EXTERIOR edges are ignored.

In [ ]:
%%writefile cvcompare.py
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from shapely.geometry import Polygon

from fpgraph import TARGET_CLASSES

CLASS_COLORS = {"wall": "#1f4e9c", "curtain_wall": "#17becf", "railing": "#8c564b", "door_single": "#d62728", "door_double": "#ff7f0e",
                "door_sliding": "#e377c2", "door_other": "#bcbd22", "window": "#2ca02c", "opening": "#7f7f7f", "stairs": "#9467bd",
                "elevator": "#111111", "other": "#d9d9d9"}


def room_polys(m):
    return [Polygon(r["polygon"]).buffer(0) for r in m["rooms"]]


def words(s):
    return set((s or "").upper().replace("/", " ").split())


def labels_agree(a, b):
    wa, wb = words(a), words(b)
    if not wa or not wb:
        return False
    return (a or "").upper().strip() == (b or "").upper().strip() or len(wa & wb) / max(len(wa | wb), 1) >= 0.5


def match_rooms(cv, ref, iou_min=0.5):
    A, B = room_polys(cv), room_polys(ref)
    iou = np.zeros((len(A), len(B)))
    for i, a in enumerate(A):
        for j, b in enumerate(B):
            if a.intersects(b):
                inter = a.intersection(b).area
                iou[i, j] = inter / max(a.area + b.area - inter, 1e-9)
    ri, ci = linear_sum_assignment(-iou) if iou.size else (np.array([], int), np.array([], int))
    pairs = [(int(i), int(j), float(iou[i, j])) for i, j in zip(ri, ci) if iou[i, j] >= iou_min]
    return pairs, iou


def match_openings(cv, ref, kind, radius=600.0):
    A = [o for o in cv["openings"] if o["kind"] == kind]
    B = [o for o in ref["openings"] if o["kind"] == kind]
    if not A or not B:
        return A, B, []
    pa, pb = np.array([o["position"] for o in A]), np.array([o["position"] for o in B])
    D = np.hypot(pa[:, None, 0] - pb[None, :, 0], pa[:, None, 1] - pb[None, :, 1])
    ri, ci = linear_sum_assignment(np.where(D <= radius, D, 1e9))
    return A, B, [(int(i), int(j)) for i, j in zip(ri, ci) if D[i, j] <= radius]


def compare_manifests(cv, ref, iou_min=0.5):
    rows = []
    add = lambda group, metric, value: rows.append({"group": group, "metric": metric, "value": value})
    add("schema", "same top-level keys and order", list(cv.keys()) == list(ref.keys()))
    for k in ("rooms", "openings", "walls", "adjacency", "corridors", "vertical_circulation", "unassigned_labels"):
        add("counts", f"{k}: cv / reference", f"{len(cv[k])} / {len(ref[k])}")
    pairs, _ = match_rooms(cv, ref, iou_min)
    add("rooms", f"matched at IoU >= {iou_min}", f"{len(pairs)} of {len(cv['rooms'])} cv, {len(ref['rooms'])} reference")
    if pairs:
        add("rooms", "mean IoU of matched", round(float(np.mean([p[2] for p in pairs])), 3))
        err = [abs(cv["rooms"][i]["area_m2"] - ref["rooms"][j]["area_m2"]) / max(ref["rooms"][j]["area_m2"], 1e-9) * 100 for i, j, _ in pairs]
        add("rooms", "median area error %", round(float(np.median(err)), 2))
        agree = [labels_agree(cv["rooms"][i]["raw_label"], ref["rooms"][j]["raw_label"]) for i, j, _ in pairs]
        add("rooms", "label agreement among matched", round(float(np.mean(agree)), 3))
    for kind in ("door", "window"):
        A, B, m = match_openings(cv, ref, kind)
        add(kind + "s", "matched / cv / reference", f"{len(m)} / {len(A)} / {len(B)}")
        if m:
            add(kind + "s", "precision, recall", f"{len(m) / len(A):.3f}, {len(m) / len(B):.3f}")
            add(kind + "s", "mean abs width error mm", round(float(np.mean([abs(A[i]["width_mm"] - B[j]["width_mm"]) for i, j in m])), 1))
    to_ref = {cv["rooms"][i]["id"]: ref["rooms"][j]["id"] for i, j, _ in pairs}
    ref_ids = set(to_ref.values())
    cv_edges = {frozenset((to_ref[e["a"]], to_ref[e["b"]])) for e in cv["adjacency"] if e["a"] in to_ref and e["b"] in to_ref}
    ref_edges = {frozenset((e["a"], e["b"])) for e in ref["adjacency"] if e["a"] in ref_ids and e["b"] in ref_ids}
    tp = len(cv_edges & ref_edges)
    p, r = tp / max(len(cv_edges), 1), tp / max(len(ref_edges), 1)
    add("adjacency", "edges among matched rooms: cv / reference / shared", f"{len(cv_edges)} / {len(ref_edges)} / {tp}")
    add("adjacency", "edge F1", round(2 * p * r / max(p + r, 1e-9), 3))
    add("labels", "unassigned: cv / reference", f"{len(cv['unassigned_labels'])} / {len(ref['unassigned_labels'])}")
    return pd.DataFrame(rows)


def plot_scene(Pw, scene, bbox=None, ax=None, max_prims=400000, lw=0.4):
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    ax = ax or plt.subplots(figsize=(10, 10))[1]
    ends = Pw["pts"][:, [0, 3]]
    keep = np.ones(len(ends), dtype=bool)
    if bbox is not None:
        x0, y0, x1, y1 = bbox
        mid = Pw["pts"].mean(1)
        keep = (mid[:, 0] >= x0) & (mid[:, 0] <= x1) & (mid[:, 1] >= y0) & (mid[:, 1] <= y1)
    sel = np.where(keep)[0]
    if len(sel) > max_prims:
        sel = np.random.default_rng(0).choice(sel, max_prims, replace=False)
    for k, name in enumerate(TARGET_CLASSES):
        s = sel[scene["cls"][sel] == k]
        if len(s):
            ax.add_collection(LineCollection(ends[s], colors=CLASS_COLORS[name], linewidths=lw if name == "other" else lw * 2.5, label=name))
    ax.autoscale()
    ax.set_aspect("equal")
    ax.legend(loc="upper right", fontsize=7, markerscale=3)
    return ax


def plot_rooms(cv, ref=None, bbox=None, ax=None):
    import matplotlib.pyplot as plt
    ax = ax or plt.subplots(figsize=(10, 10))[1]
    for m, color, label in ((ref, "#d62728", "classical"), (cv, "#1f77b4", "cv_model")):
        if m is None:
            continue
        for k, r in enumerate(m["rooms"]):
            xy = np.array(r["polygon"])
            ax.plot(xy[:, 0], xy[:, 1], color=color, lw=0.8, label=label if k == 0 else None)
    if bbox is not None:
        ax.set_xlim(bbox[0], bbox[2])
        ax.set_ylim(bbox[1], bbox[3])
    ax.set_aspect("equal")
    ax.legend(loc="upper right")
    return ax


def plan_bbox(manifest, pad=3000.0):
    xy = np.vstack([np.array(r["polygon"]) for r in manifest["rooms"]])
    return (*(xy.min(0) - pad), *(xy.max(0) + pad))

### 2f. `cvpipeline.py`: one call, DXF in, manifest out
The same steps as the notebook, in one function, for batch runs and for the FastAPI service later. It takes a loaded model, so the checkpoint is read once and reused across floors.

In [ ]:
%%writefile cvpipeline.py
import json
import time
from pathlib import Path

import cvexport
import cvadapter
import gnninfer


def run_floor(dxf_path, model, phase_a, lut, stuff, device, work_dir="cv_work", thr=0.5, floor_only=True, floor_block=None, mm_factor=None,
              node_budget=12000, max_tiles=8, amp=False, out_path=None):
    t0 = time.time()
    work = Path(work_dir)
    work.mkdir(parents=True, exist_ok=True)
    svg = work / (Path(dxf_path).stem.replace(" ", "_") + ".svg")
    doc, top, ex, info = cvexport.export_svg(dxf_path, svg, floor_only=floor_only, floor_block=floor_block, mm_factor=mm_factor)
    cfg = gnninfer.graph_config(phase_a)
    P = gnninfer.read_primitives(svg, cfg)
    tiles = gnninfer.build_tiles(P, cfg, lut, stuff)
    outs = gnninfer.predict_tiles(model, tiles, device, node_budget, max_tiles, amp)
    merged = gnninfer.merge_tiles(len(P["ptype"]), tiles, outs)
    scene = gnninfer.decode_scene(P, merged, thr)
    Pw = cvadapter.to_wcs(P, info["origin_mm"])
    manifest, r = cvadapter.run_cv(svg, info, Pw, scene, top=top)
    manifest["summary"]["cv"]["total_elapsed_s"] = round(time.time() - t0, 1)
    if out_path:
        Path(out_path).write_text(json.dumps(manifest, indent=1, default=float))
    return manifest

## 3. Load the model
All modules are (re)imported from disk, so re-running the writefile cells above always takes effect. The model is rebuilt from the config stored inside the checkpoint, so it cannot drift from what was trained.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

for _m in ("fpgraph", "tracka_core", "indlabel", "gnnmodel", "gnndata", "gnneval", "cvexport", "gnninfer", "cvadapter", "cvcompare", "cvpipeline"):
    importlib.invalidate_caches()
    sys.modules[_m] = importlib.reload(sys.modules[_m]) if _m in sys.modules else importlib.import_module(_m)

import fpgraph
import tracka_core as core
import indlabel
import cvexport, gnninfer, cvadapter, cvcompare, cvpipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PHASE_A = json.loads(Path("config.json").read_text())
LUT = fpgraph.build_label_map(PHASE_A["raw_class_names"], PHASE_A["harmonization"], PHASE_A["targets"])
STUFF = fpgraph.stuff_mask(PHASE_A["targets"])
CFG = gnninfer.graph_config(PHASE_A)

model, CK = gnninfer.load_model(Path(CKPT_SUBDIR) / "best.pt", device)
THR = HEAD2_THR if HEAD2_THR is not None else gnninfer.read_threshold(Path(CKPT_SUBDIR) / "head2_threshold.json")
print("device:", device, "| classes:", gnninfer.CLASSES)
print("model parameters:", sum(p.numel() for p in model.parameters()), "| Head 2 threshold:", THR)

## 4. The drawing: DWG to DXF, and which floor
**Get the file.** With `DWG_PATH = None` a file picker opens (Colab). The file is copied to `dwg_in/`.

**Convert.** `indlabel.convert_dwgs` runs ODA File Converter headless (DWG to DXF, version ACAD2018) into `cv_work/dxf/`. If the file is already a DXF it is used as it is. The converter prints nothing; success means the `.dxf` exists.

**Choose the floor.** The DXF is read once and the INSERTs in modelspace are listed with the number of entities in their blocks. Stage 2a takes the biggest one. If the DWG holds several floors, two or more rows will have comparable sizes and a warning is printed; put the name of the one you want in `FLOOR_BLOCK` and re-run this cell.

In [ ]:
if DWG_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    DWG_PATH = next(iter(uploaded))
src = Path(DWG_PATH)
assert src.exists(), f"{src} not found"

if src.suffix.lower() == ".dwg":
    stage = Path("dwg_in")
    shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir()
    shutil.copy(src, stage / src.name)
    DXF_DIR = WORK_DIR / "dxf"
    shutil.rmtree(DXF_DIR, ignore_errors=True)
    print(indlabel.convert_dwgs(stage, DXF_DIR, "ODAFileConverter"))
    DXF_PATH = DXF_DIR / (src.stem + ".dxf")
else:
    DXF_PATH = src
assert DXF_PATH.exists(), "DXF not produced: check the ODA install cell and that the DWG opens in AutoCAD or ODA"
print("DXF:", DXF_PATH, f"{DXF_PATH.stat().st_size / 1e6:.1f} MB")

t0 = time.time()
DOC, READ_STATUS = indlabel.read_dxf(DXF_PATH)
BLOCKS = pd.DataFrame(cvexport.list_floor_blocks(DOC))
display(BLOCKS)
print(f"read {READ_STATUS} in {time.time() - t0:.0f}s | insunits header:", DOC.header.get("$INSUNITS"))
if FLOOR_BLOCK is None and len(BLOCKS) > 1 and BLOCKS["entities"].iloc[1] > 0.2 * BLOCKS["entities"].iloc[0]:
    print("WARNING: several large blocks; this may be a multi-floor drawing. Set FLOOR_BLOCK to the one you want.")

## 5. DXF to a label-free SVG
`export_svg` explodes the floor block, decides the unit factor, scales everything to mm and writes the SVG. Look at three things in the printout:
- **`mm_factor`** should be 1.0 for a drawing in mm, and the unit `status` should be `confirmed` (door swing and wall thickness both agree with the factor). Anything else means the model would see the drawing at the wrong scale and every later number would be wrong. If it is wrong, set `MM_FACTOR`.
- **`n_text`**: the room tags and other texts that step 8 will use for names.
- **`floor_block`**: the block that was exploded.

The silver-label SVG is written **after** the clean one and is used only in steps 6 and 7.

In [ ]:
SVG_PATH = WORK_DIR / "floor.svg"
SILVER_PATH = WORK_DIR / "floor_silver.svg"

doc, top, ex, INFO = cvexport.export_svg(DXF_PATH, SVG_PATH, floor_only=FLOOR_ONLY, floor_block=FLOOR_BLOCK, mm_factor=MM_FACTOR, doc=DOC)
cvexport.export_silver_svg(ex, SILVER_PATH, PHASE_A["raw_class_names"])
print({k: INFO[k] for k in ("source_file", "floor_block", "n_prims", "n_text", "mm_factor", "origin_mm", "size_mm")})
print("unit inference:", INFO["units"])
print("clean SVG:", SVG_PATH.stat().st_size // 1_000_000, "MB | explode errors:", INFO["explode_errors"])

## 6. Tile, predict, merge, decode
**Tile.** `read_primitives` parses the SVG (and splits lines longer than 2 m, as in training); `build_tiles` cuts 10 m tiles with 1 m overlap and builds each graph, keeping every tile's list of global primitive ids.

**Predict.** Tiles are packed into batches under a node budget and run through the GNN. `src = 1` marks them as Indian-style input, which only the fine-tuning loss used; the forward pass does not depend on it.

**Merge and decode.** Averaged logits per primitive and per edge, then instances on the whole drawing at `THR`.

Trace for one door that straddles a tile border: tile A holds 4 of its 5 primitives and tile B holds 4 (3 shared). The 3 shared primitives get the mean of A's and B's logits. The edges A-A, B-B and the shared ones are all in the merged edge list, keyed by global ids, so a single connected component links all 5 primitives, and the length-weighted vote gives it one class.

Read the counts: **uncovered primitives** should be near 0 (a few hundred in a dense area would mean the 4,096-node tile cap is cutting nodes), and the class counts should look like a floor plan: mostly `wall` and `other`, with door and window instances counted per class.

In [ ]:
t0 = time.time()
P = gnninfer.read_primitives(SVG_PATH, CFG)
tiles = gnninfer.build_tiles(P, CFG, LUT, STUFF)
sizes = np.array([len(t["y"]) for t in tiles])
print(f"{len(P['ptype']):,} primitives -> {len(tiles)} tiles (nodes per tile: median {int(np.median(sizes))}, max {sizes.max()}) in {time.time() - t0:.0f}s")

t0 = time.time()
outs = gnninfer.predict_tiles(model, tiles, device, NODE_BUDGET, MAX_TILES, USE_AMP)
merged = gnninfer.merge_tiles(len(P["ptype"]), tiles, outs)
SCENE = gnninfer.decode_scene(P, merged, THR)
Pw = cvadapter.to_wcs(P, INFO["origin_mm"])
print(f"model + merge + decode: {time.time() - t0:.0f}s on {device}")

names = np.array(gnninfer.CLASSES)
counts = np.bincount(SCENE["cls"], minlength=len(names))
inst = pd.Series(names[SCENE["seg_cls"]]).value_counts()
display(pd.DataFrame({"class": names, "primitives": counts, "instances": [int(inst.get(n, 0)) for n in names]}))
print("uncovered primitives:", int((~SCENE["covered"]).sum()), "| primitives seen by 2+ tiles:", int((merged["n_tiles_per_prim"] > 1).sum()))
print("probability of the final class: median", round(float(np.median(SCENE["prob"][SCENE["covered"]])), 3),
      "| below 0.7:", int((SCENE["prob"][SCENE["covered"]] < 0.7).sum()))

## 7. Agreement with the rule-based (silver) labels on this drawing
The silver labels are the Phase B rules applied to this DXF's layer and block names. This is **agreement with the rules, not accuracy**: on a drawing from the training office it is optimistic, and on a drawing whose layers the rules do not recognise the silver labels are mostly `other` and the table says little. Read `silver_prims` first.

It is scored with the very functions used in Phase C/D (`f1_from_conf`, `panoptic_counts`, `pq_from_counts`). Classes with 0 silver primitives show F1 0: that is "no examples", not "failed".

In [ ]:
P_GT = gnninfer.read_primitives(SILVER_PATH, CFG)
assert len(P_GT["ptype"]) == len(P["ptype"]), "clean and silver SVG must have the same primitives"
AGREE, AGREE_SUMMARY = gnninfer.silver_agreement(P_GT, SCENE, LUT, STUFF)
display(AGREE)
print({k: round(v, 4) for k, v in AGREE_SUMMARY.items()})

### 7.1 Look at it
Left: the model's class for every primitive. Right: the silver labels. Systematic differences show up immediately (every door arc read as a window, a wall class missing in a zone). `ZOOM = None` shows the whole drawing; set `ZOOM = (x0, y0, x1, y1)` in drawing coordinates to inspect one area. At most 400,000 primitives are drawn.

In [ ]:
ZOOM = None

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
cvcompare.plot_scene(Pw, SCENE, bbox=ZOOM, ax=axes[0])
cvcompare.plot_scene(Pw, gnninfer.scene_from_silver(P_GT, LUT, STUFF), bbox=ZOOM, ax=axes[1])
axes[0].set_title("GNN prediction")
axes[1].set_title("silver labels (Phase B rules)")
plt.tight_layout()
plt.show()

## 8. The manifest from the model
`run_cv` swaps four inputs of the Track A core for the model's, runs `core.run()` and `core.build_manifest()`, then sets `source = "cv_model"` and the confidences, and validates the result with the **same Pydantic model** Stage 2a uses (`ManifestM`). If this cell finishes, the file has the classical schema.

Confidence is advisory, as planned: for an opening, the minimum softmax probability over its primitives; for a wall segment, the minimum over the wall primitives it covers; for a room, the minimum over the boundary primitives within 30 mm of its outline. It is then capped by Track A's own confidence (0.5 for a self-looping door, 0.6 for an unlabelled space, and so on). The model-only numbers are kept next to it as `cv_*` fields.

The file is written to `OUT_PATH`, by default `manifest_<drawing name>_cv.json`, and Colab's file browser (or `files.download`) gets it from there.

In [ ]:
MANIFEST, R = cvadapter.run_cv(SVG_PATH, INFO, Pw, SCENE, top=top)
OUT = Path(OUT_PATH) if OUT_PATH else Path(f"manifest_{DXF_PATH.stem.replace(' ', '_')}_cv.json")
OUT.write_text(json.dumps(MANIFEST, indent=1, default=float))
print("written", OUT, f"{OUT.stat().st_size / 1e6:.2f} MB | keys:", list(MANIFEST.keys()))
print(json.dumps({k: v for k, v in MANIFEST["summary"].items() if k != "cv"}, indent=1))
print(json.dumps({k: v for k, v in MANIFEST["summary"]["cv"].items() if k != "symbols"}, indent=1))
print("stairs / elevator symbols found by the model:", [(s["class"], s["n_prims"]) for s in MANIFEST["summary"]["cv"]["symbols"]][:20])

### 8.1 Download
In Colab this offers the manifest as a download; elsewhere it does nothing.

In [ ]:
try:
    from google.colab import files
    files.download(str(OUT))
except Exception as e:
    print("not on Colab or download blocked:", type(e).__name__, "| file is at", OUT.resolve())

### 8.2 Rooms on the plan
The rooms the model's walls, doors and windows produced, drawn over the drawing's extent. If a classical manifest is given (`REFERENCE_MANIFEST`), its rooms are drawn in red for comparison.

In [ ]:
REF = None
if REFERENCE_MANIFEST and Path(REFERENCE_MANIFEST).exists():
    REF = json.loads(Path(REFERENCE_MANIFEST).read_text())
    core.ManifestM.model_validate(REF)
BOX = cvcompare.plan_bbox(REF or MANIFEST)
fig, ax = plt.subplots(figsize=(14, 14))
cvcompare.plot_rooms(MANIFEST, REF, bbox=BOX, ax=ax)
plt.show()
print("classical reference:", "loaded" if REF else "none given")

### 8.3 What a reviewer would look at first
The least confident records of the CV manifest. Records below 0.7 form the review queue, in the spirit of "CV-derived flags are advisory".

In [ ]:
rows = []
for kind in ("rooms", "openings", "walls"):
    for rec in MANIFEST[kind]:
        rows.append({"kind": kind[:-1], "id": rec["id"], "label": rec.get("raw_label") or rec.get("cv_class") or "", "confidence": rec["confidence"]})
queue = pd.DataFrame(rows)
print("records below 0.7:", queue.groupby("kind")["confidence"].agg(lambda c: int((c < 0.7).sum())).to_dict(), "of", queue.groupby("kind").size().to_dict())
display(queue.sort_values("confidence").head(15))
display(queue.groupby("kind")["confidence"].describe().round(3))

## 9. Optional: compare with a classical manifest
Runs only if `REFERENCE_MANIFEST` points to a Stage 2a manifest of the **same floor**. The classical manifest is a reference, not ground truth.

1. **schema**: must be `True`.
2. **counts**: rooms, openings, walls and adjacency edges, cv against classical. Wall counts can differ legitimately: the classical walls come from layer `A-WALL` only, whereas here columns are also predicted as `wall`.
3. **rooms**: how many rooms line up (IoU of at least 0.5), mean IoU, median area error, and whether the names agree.
4. **doors / windows**: matched within 600 mm of each other, precision, recall and width error.
5. **adjacency**: edge F1 among rooms that matched on both sides.

With `RUN_ORACLE = True` the silver labels are also pushed through the same adapter as if they were perfect model output (probability 1), and shown beside the model column. The gap between **oracle and classical** belongs to the adapter and the label rules; the gap between **model and oracle** belongs to the model. It costs one more run of the geometry core.

In [ ]:
if REF is None:
    print("no REFERENCE_MANIFEST given: nothing to compare")
else:
    cmp_model = cvcompare.compare_manifests(MANIFEST, REF)
    if RUN_ORACLE:
        ORACLE, _ = cvadapter.run_cv(SVG_PATH, INFO, Pw, gnninfer.scene_from_silver(P_GT, LUT, STUFF), top=top)
        table = cmp_model.merge(cvcompare.compare_manifests(ORACLE, REF), on=["group", "metric"], suffixes=("_model", "_oracle"))
    else:
        table = cmp_model
    pd.set_option("display.max_colwidth", 80)
    display(table)

## 10. One-call version
`cvpipeline.run_floor` runs steps 5 to 8 in one call, from a DXF, and takes a model that is already loaded, so a batch of floors (or the later FastAPI service) loads the checkpoint once. It repeats the whole computation, so it is off by default (`RUN_ONE_CALL`). When it is on, it must give the same counts as step 8.

In [ ]:
if RUN_ONE_CALL:
    again = cvpipeline.run_floor(DXF_PATH, model, PHASE_A, LUT, STUFF, device, work_dir=WORK_DIR / "one_call", thr=THR,
                                 floor_only=FLOOR_ONLY, floor_block=FLOOR_BLOCK, mm_factor=MM_FACTOR, node_budget=NODE_BUDGET,
                                 max_tiles=MAX_TILES, amp=USE_AMP, out_path=WORK_DIR / "one_call_manifest.json")
    print({k: again["summary"][k] for k in ("rooms", "doors", "windows", "walls", "adjacency_edges")},
          "| same as step 8:", {k: again["summary"][k] == MANIFEST["summary"][k] for k in ("rooms", "doors", "windows", "walls")})
else:
    print("skipped (RUN_ONE_CALL = False)")

## Known limits (read before trusting a CV flag)
- **DWG to DXF.** The converter writes DXF version ACAD2018. Stage 2a's own DXF came from an earlier stage (AC1015), so a small difference in how some entities are written is possible. Units are not taken from the header alone: the exporter checks them against door swings and wall thickness.
- **Several floors in one DWG.** Only one floor block is processed (`FLOOR_BLOCK`). Run the notebook once per floor.
- **Elevators.** The Indian training set labels lifts as `other` (no lift blocks or layers in the source drawings), so the fine-tuned model rarely predicts `elevator`. `vertical_circulation` comes from room names, as in Track A, and the model's own stairs and elevator symbols are only listed in `summary.cv.symbols`.
- **`curtain_wall`, `door_other`, `opening`** have no Indian training examples. Doorless openings are not found by the model. As in Track A they appear only where a tag-guided wall gap splits a space (`passages`).
- **Room names are a text heuristic.** The SVG has no layers, so a name is any text with at least 3 letters that is not a door or window code, and a size such as `3650X4170` is attached to the nearest name within 1.5 m. Titles, grid names and notes inside a room would become labels. Text positions use each text's insertion point, so centred or right-aligned text can sit slightly off the tag position Track A uses.
- **No dimensions.** The exporter does not keep `DIMENSION` entities, so `dimensions` is empty and the dimension cross-check of `scale_resolution` has nothing to test. The opening-name check is empty too (no block names). `scale_resolution.status` rests on the room-size check alone, and the unit inference is recorded under `scale_resolution.unit_inference`.
- **Balconies.** `A-BALCONY` is unlabelled in Phase B (so `other`), whereas Track A treats it as a boundary. Balcony areas can therefore differ from a classical manifest.
- **Door and window widths.** With no block name, the nominal width is `null`. Widths come from the jamb gap, then the swing radius, then the extent of the predicted primitives.
- **Silver labels, same drafter.** Agreement in step 7 is against rule-based labels; on a floor from the training office it is optimistic and says nothing about other offices.
- **Confidence is a minimum**, so one uncertain primitive near a room lowers the whole room. Use the `cv_*_conf_mean` fields next to it when a room looks over-penalised.